<a href="https://colab.research.google.com/github/AntonDozhdikov/politpredict/blob/main/adaptive_foresight_pipeline_work_final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🌐 Adaptive Multi-Agent Foresight Pipeline
### From Futures Cone to NARL-DGM: A Computational Framework for Strategic Foresight


> **Dataset:** World Development Indicators + Worldwide Governance Indicators (World Bank, open data)  
> **Hardware:** Google Colab (T4 GPU) | Estimated runtime: **2–3 hours**  
> **Reproducibility:** All seeds fixed · All artifacts auto-saved to `output/`

---

| # | Section | Est. Time |
|---|---------|-----------|
| 1 | Setup — install, imports, config, utils | < 2 min |
| 2 | Data pipeline — WDI + WGI, cleaning, engineering | ~10 min |
| 3 | **Experiment 1 — Baseline** VAR + scenario clustering | ~5 min |
| 4 | **Experiment 2 — MARL** CTDE-MAPPO multi-country | ~60 min |
| 5 | **Experiment 3 — NARL-DGM** Meta-meta + adaptive reward | ~90 min |
| 6 | Comparison, publication figures, artifact save | ~5 min |

---
> ⚠️ **Note:** If World Bank API is unavailable, the pipeline auto-falls back to cached CSV (if present) or synthetic panel for testing.


In [1]:
# ─── CELL 1: INSTALL DEPENDENCIES ────────────────────────────────────────
%%capture
import subprocess, sys

_pkgs = [
    "wbgapi>=1.0.12", "statsmodels>=0.14", "scipy>=1.11",
    "scikit-learn>=1.3", "seaborn>=0.13", "tqdm>=4.66",
    "nbformat", "joblib"
]
for _p in _pkgs:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", _p], check=False)

print("✅ Dependencies installed")


In [2]:
# ─── CELL 2: IMPORTS + GPU SETUP ─────────────────────────────────────────
import os, sys, time, json, random, warnings, zipfile, shutil
from pathlib import Path
from contextlib import contextmanager
from dataclasses import dataclass, field
from typing import Dict, List, Tuple, Optional
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from tqdm.auto import tqdm, trange

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.distributions import Normal
from torch.optim.lr_scheduler import CosineAnnealingLR

from sklearn.preprocessing import RobustScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.impute import SimpleImputer
from scipy import stats
import statsmodels.api as sm

# ── GPU / CPU ──────────────────────────────────────────────────────────────
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🖥  Device  : {DEVICE}")
if torch.cuda.is_available():
    g = torch.cuda.get_device_properties(0)
    print(f"   GPU     : {g.name}")
    print(f"   VRAM    : {g.total_memory / 1e9:.1f} GB")
else:
    print("   ⚡ Running on CPU — MARL will be slower; reduce N_EPISODES if needed")
print(f"   PyTorch : {torch.__version__}")
print(f"   pandas  : {pd.__version__}")


🖥  Device  : cuda
   GPU     : Tesla T4
   VRAM    : 15.6 GB
   PyTorch : 2.10.0+cu128
   pandas  : 2.2.2


In [3]:
# ─── CELL 3: MASTER CONFIG ───────────────────────────────────────────────
from dataclasses import dataclass, field
from typing import Dict, List

@dataclass
class Config:
    # ── Reproducibility ────────────────────────────────────────────────────
    SEED: int = 42

    # ── Data ───────────────────────────────────────────────────────────────
    YEAR_START: int = 1996
    YEAR_END: int = 2023  # 2024 often incomplete in WDI
    MIN_COVERAGE: float = 0.55  # min share of non-NaN per country
    MAX_COUNTRIES: int = 50     # top-N by coverage (for fast Colab run)

    # ── WDI indicator codes → feature names ────────────────────────────────
    INDICATORS_WDI: Dict[str, str] = field(default_factory=lambda: {
        # — Macroeconomics —
        "gdp_growth": "NY.GDP.MKTP.KD.ZG",
        "gdp_pc_usd": "NY.GDP.PCAP.CD",
        "gdp_pc_growth": "NY.GDP.PCAP.KD.ZG",
        "inflation": "FP.CPI.TOTL.ZG",
        "unemployment": "SL.UEM.TOTL.ZS",
        "gross_capital_form": "NE.GDI.TOTL.ZS",
        "govt_consumption": "NE.CON.GOVT.ZS",
        # — Trade & Finance —
        "exports_gdp": "NE.EXP.GNFS.ZS",
        "imports_gdp": "NE.IMP.GNFS.ZS",
        "trade_gdp": "NE.TRD.GNFS.ZS",
        "fdi_gdp": "BX.KLT.DINV.WD.GD.ZS",
        "manufacturing_gdp": "NV.IND.MANF.ZS",
        "hitech_exports": "TX.VAL.TECH.MF.ZS",
        # — R&D & Science —
        "rd_gdp": "GB.XPD.RSDV.GD.ZS",
        "researchers_pm": "SP.POP.SCIE.RD.P6",
        "journal_articles": "IP.JRN.ARTC.SC",
        "patents_residents": "IP.PAT.RESD",
        # — Digital —
        "internet_users": "IT.NET.USER.ZS",
        "mobile_subs": "IT.CEL.SETS.P2",
        "broadband": "IT.NET.BBND.P2",
        "secure_servers_pm": "IT.NET.SECR.P6",
        # — Demographics & Human Capital —
        "life_expectancy": "SP.DYN.LE00.IN",
        "infant_mortality": "SP.DYN.IMRT.IN",
        "school_secondary": "SE.SEC.ENRR",
        "school_tertiary": "SE.TER.ENRR",
        "edu_gdp": "SE.XPD.TOTL.GD.ZS",
        "labor_participation": "SL.TLF.CACT.ZS",
        "female_labor": "SL.TLF.CACT.FE.ZS",
        "gini_index": "SI.POV.GINI",
        "pop_growth": "SP.POP.GROW",
        "urban_pop_pct": "SP.URB.TOTL.IN.ZS",
        # — Energy & Environment —
        "co2_pc": "EN.ATM.CO2E.PC",
        "renewable_energy": "EG.FEC.RNEW.ZS",
        "electricity_access": "EG.ELC.ACCS.ZS",
        "energy_use_pc": "EG.USE.PCAP.KG.OE",
        "forest_area_pct": "AG.LND.FRST.ZS",
        "nat_resources_gdp": "NY.GDP.TOTL.RT.ZS",
    })

    # ── WGI governance codes ────────────────────────────────────────────────
    WGI_CODES: Dict[str, str] = field(default_factory=lambda: {
        "wgi_voice": "VA.EST",
        "wgi_stability": "PV.EST",
        "wgi_govt_eff": "GE.EST",
        "wgi_reg_qual": "RQ.EST",
        "wgi_rule_law": "RL.EST",
        "wgi_corruption": "CC.EST",
    })

    # ── Environment ─────────────────────────────────────────────────────────
    STATE_DIM: int = 42   # updated later after feature engineering
    ACTION_DIM: int = 8   # policy portfolio dimensions
    EPISODE_LEN: int = 10 # steps (years) per episode

    # Action dimensions (policy portfolio):
    # [0] rd_intensity — R&D & AI investment share
    # [1] edu_intensity — education & human capital spend
    # [2] social_prot   — social protection & inequality
    # [3] regulation    — regulatory stringency
    # [4] green_trans   — green/energy transition
    # [5] trade_open    — trade & FDI openness
    # [6] reserves      — resilience/fiscal reserves
    # [7] ai_regime     — AI governance regime weight (→ softmax over 6 regimes)

    # ── Network ──────────────────────────────────────────────────────────────
    HIDDEN_DIM: int = 256
    GRU_LAYERS: int = 1
    DROPOUT: float = 0.10

    # ── MARL training (CTDE-MAPPO) ──────────────────────────────────────────
    N_EPISODES: int = 2500          # немного больше эпизодов, чем было
    BATCH_SIZE: int = 256
    LR_ACTOR: float = 2.5e-4        # чуть меньше, для более стабильного PPO
    LR_CRITIC: float = 8e-4         # критик учится немного быстрее
    GAMMA: float = 0.99
    GAE_LAMBDA: float = 0.95
    CLIP_EPSILON: float = 0.20
    ENTROPY_COEF: float = 0.015     # чуть сильнее exploration
    VALUE_COEF: float = 0.50
    MAX_GRAD_NORM: float = 0.50
    N_EPOCHS_PPO: int = 4
    LOG_INTERVAL: int = 100

    # ── Early stopping ───────────────────────────────────────────────────────
    PATIENCE: int = 900            # подольше ждём, чтобы не срезать хвост
    MIN_DELTA: float = 5e-5        # требуемая прибавка reward

    # ── NARL-DGM meta-learning ───────────────────────────────────────────────
    META_LR: float = 5e-4          # мета-контроллер учится осторожнее
    META_UPDATE_INTERVAL: int = 50
    META_META_INTERVAL: int = 200
    REWARD_HACK_THRESHOLD: float = 0.20
    N_REWARD_COMPONENTS: int = 7
    N_META_EPISODES: int = 1200    # немного больше meta-эпизодов

    # ── Crisis module ────────────────────────────────────────────────────────
    CRISIS_INJECTION_PROB: float = 0.15
    TAIL_RISK_PROB: float = 0.02

    # AI Governance regimes
    AI_REGIME_NAMES: List[str] = field(default_factory=lambda: [
        "pluralistic", "sovereign", "regulated",
        "market_driven", "multilateral", "hybrid"
    ])

    # ── Paths ─────────────────────────────────────────────────────────────────
    DATA_DIR: str = "data"
    OUTPUT_DIR: str = "output"
    CHECKPOINT_DIR: str = "output/checkpoints"


CFG = Config()

# Seed everything
def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

set_seed(CFG.SEED)

# Create dirs
for _d in [CFG.DATA_DIR, CFG.OUTPUT_DIR, CFG.CHECKPOINT_DIR]:
    Path(_d).mkdir(parents=True, exist_ok=True)

print("✅ Config initialized")
print(f" WDI indicators : {len(CFG.INDICATORS_WDI)}")
print(f" WGI dimensions : {len(CFG.WGI_CODES)}")
print(f" State dim       : {CFG.STATE_DIM}")
print(f" Action dim      : {CFG.ACTION_DIM}")
print(f" Max countries   : {CFG.MAX_COUNTRIES}")
print(f" MARL episodes   : {CFG.N_EPISODES}")
print(f" META episodes   : {CFG.N_META_EPISODES}")

✅ Config initialized
 WDI indicators : 37
 WGI dimensions : 6
 State dim       : 42
 Action dim      : 8
 Max countries   : 50
 MARL episodes   : 2500
 META episodes   : 1200


In [4]:
# ─── CELL 4: UTILITIES — Timer, Logger, EarlyStopping, ArtifactManager ───

import time
from contextlib import contextmanager

# ── Timer context manager ─────────────────────────────────────────────────
class Timer:
    """Lightweight wall-clock timer with nested support."""
    _stack = []
    _log   = {}

    def __init__(self, name: str, verbose: bool = True):
        self.name    = name
        self.verbose = verbose
        self._start  = None

    def __enter__(self):
        self._start = time.perf_counter()
        Timer._stack.append(self.name)
        return self

    def __exit__(self, *_):
        elapsed = time.perf_counter() - self._start
        Timer._log[self.name] = elapsed
        Timer._stack.pop()
        indent = "  " * len(Timer._stack)
        if self.verbose:
            print(f"{indent}⏱  {self.name}: {elapsed:.2f}s")

    @classmethod
    def summary(cls):
        print("\n─── Timing Summary ───────────────────────────────")
        for k, v in cls._log.items():
            print(f"  {k:<40s}: {v:7.2f}s")

# ── Early Stopping ────────────────────────────────────────────────────────
class EarlyStopping:
    """Monitor a scalar metric; stop training when no improvement."""
    def __init__(self, patience: int = 200, min_delta: float = 1e-4,
                 mode: str = "max", restore_best: bool = True):
        self.patience     = patience
        self.min_delta    = min_delta
        self.mode         = mode
        self.restore_best = restore_best
        self.best_value   = -np.inf if mode == "max" else np.inf
        self.best_state   = None
        self.counter      = 0
        self.triggered    = False

    def __call__(self, value: float, model_state=None) -> bool:
        improved = (self.mode == "max" and value > self.best_value + self.min_delta) or                    (self.mode == "min" and value < self.best_value - self.min_delta)
        if improved:
            self.best_value = value
            self.counter    = 0
            if model_state is not None:
                import copy
                self.best_state = copy.deepcopy(model_state)
        else:
            self.counter += 1
        if self.counter >= self.patience:
            self.triggered = True
        return self.triggered

# ── Logger ────────────────────────────────────────────────────────────────
class ExperimentLogger:
    """Lightweight experiment logger to CSV."""
    def __init__(self, name: str):
        self.name    = name
        self.history = []
        self._path   = Path(CFG.OUTPUT_DIR) / f"log_{name}.csv"

    def log(self, **kwargs):
        self.history.append(kwargs)

    def save(self):
        pd.DataFrame(self.history).to_csv(self._path, index=False)
        print(f"   📋 Log saved → {self._path}")

    def to_df(self) -> pd.DataFrame:
        return pd.DataFrame(self.history)

# ── Artifact Manager ──────────────────────────────────────────────────────
class ArtifactManager:
    """Save publication-ready figures + tables; build final archive."""
    def __init__(self):
        self._saved = []

    def save_fig(self, fig, name: str, dpi: int = 180):
        p = Path(CFG.OUTPUT_DIR) / f"{name}.pdf"
        fig.savefig(p, dpi=dpi, bbox_inches="tight")
        p2 = Path(CFG.OUTPUT_DIR) / f"{name}.png"
        fig.savefig(p2, dpi=dpi, bbox_inches="tight")
        plt.close(fig)
        self._saved.append(str(p))
        print(f"   🖼  Figure saved → {p.name}")

    def save_table(self, df: pd.DataFrame, name: str):
        p = Path(CFG.OUTPUT_DIR) / f"{name}.csv"
        df.to_csv(p, index=True)
        p2 = Path(CFG.OUTPUT_DIR) / f"{name}.tex"
        df.to_latex(p2, float_format="%.3f")
        self._saved.append(str(p))
        print(f"   📄 Table saved  → {p.name}")

    def build_archive(self):
        archive = shutil.make_archive("foresight_results", "zip", CFG.OUTPUT_DIR)
        print(f"\n📦 Archive ready → {archive}")
        return archive

ARTS = ArtifactManager()
print("✅ Utilities ready  (Timer · EarlyStopping · ExperimentLogger · ArtifactManager)")


✅ Utilities ready  (Timer · EarlyStopping · ExperimentLogger · ArtifactManager)


## 📥 Section 1 — Data Pipeline

In [8]:
# CELL 5 LOCAL DATA LOAD: WDI CSV + WGI XLSX (replace API download cell)

from pathlib import Path
import pandas as pd
import numpy as np

WDI_SOURCE = Path(CFG.DATA_DIR) / "wdi_panel_raw.csv"
WGI_SOURCE = Path(CFG.DATA_DIR) / "wgidataset_with_sourcedata-2025.xlsx"

WDI_CACHE = Path(CFG.DATA_DIR) / "wdi_panel_raw.csv"
WGI_CACHE = Path(CFG.DATA_DIR) / "wgi_panel_raw.csv"

if not WDI_SOURCE.exists():
    raise FileNotFoundError(f"WDI file not found: {WDI_SOURCE}")

if not WGI_SOURCE.exists():
    raise FileNotFoundError(f"WGI file not found: {WGI_SOURCE}")

def load_wdi_local(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path)

    df.columns = [c.strip() for c in df.columns]

    required_keys = ["country_code", "year"]
    for c in required_keys:
        if c not in df.columns:
            raise ValueError(f"WDI missing required column: {c}")

    if "country_name" not in df.columns:
        df["country_name"] = df["country_code"]

    df["country_code"] = df["country_code"].astype(str).str.strip()
    df["country_name"] = df["country_name"].astype(str).str.strip()
    df["year"] = pd.to_numeric(df["year"], errors="coerce").astype("Int64")

    wanted_features = [c for c in CFG.INDICATORS_WDI.keys() if c in df.columns]
    missing_features = [c for c in CFG.INDICATORS_WDI.keys() if c not in df.columns]

    keep_cols = ["country_code", "country_name", "year"] + wanted_features
    df = df[keep_cols].copy()

    for c in wanted_features:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    df = df[df["year"].notna()].copy()
    df["year"] = df["year"].astype(int)
    df = df[(df["year"] >= CFG.YEAR_START) & (df["year"] <= CFG.YEAR_END)].copy()

    df = df.drop_duplicates(subset=["country_code", "year"], keep="last")
    df = df.sort_values(["country_code", "year"]).reset_index(drop=True)

    print(f"WDI loaded: {df.shape}")
    print(f"WDI countries: {df['country_code'].nunique()}")
    print(f"WDI years: {df['year'].min()}-{df['year'].max()}")
    print(f"WDI features present: {len(wanted_features)} / {len(CFG.INDICATORS_WDI)}")
    if missing_features:
        print("WDI missing features:", missing_features)

    return df

def detect_wgi_sheet_map(xls: pd.ExcelFile) -> dict:
    sheet_map = {}
    target_codes = {
        "va": "wgivoice",
        "pv": "wgistability",
        "ge": "wgigovteff",
        "rq": "wgiregqual",
        "rl": "wgirulelaw",
        "cc": "wgicorruption",
    }

    for s in xls.sheet_names:
        sl = s.strip().lower()
        if sl in target_codes:
            sheet_map[sl] = s

    if len(sheet_map) < 6:
        print("Detected WGI sheets:", xls.sheet_names)
        print("Matched WGI sheets:", sheet_map)

    return sheet_map

def find_wgi_columns(df0: pd.DataFrame):
    cols = {c.strip(): c for c in df0.columns}

    code_col = None
    year_col = None
    est_col = None
    dim_col = None
    name_col = None

    for c in df0.columns:
        cl = c.strip().lower()
        if cl == "economy (code)":
            code_col = c
        elif cl == "economy (name)":
            name_col = c
        elif cl == "year":
            year_col = c
        elif "governance estimate" in cl:
            est_col = c
        elif cl == "governance dimension":
            dim_col = c

    if code_col is None or year_col is None or est_col is None:
        raise ValueError(
            f"Could not detect required WGI columns. Found columns: {list(df0.columns)}"
        )

    return code_col, name_col, year_col, dim_col, est_col

def load_wgi_local(path: Path) -> pd.DataFrame:
    xls = pd.ExcelFile(path)
    sheet_map = detect_wgi_sheet_map(xls)

    target_codes = {
        "va": "wgivoice",
        "pv": "wgistability",
        "ge": "wgigovteff",
        "rq": "wgiregqual",
        "rl": "wgirulelaw",
        "cc": "wgicorruption",
    }

    frames = []

    for short_code, out_col in target_codes.items():
        if short_code not in sheet_map:
            print(f"WARNING: WGI sheet for {short_code} not found, skipping")
            continue

        sheet_name = sheet_map[short_code]
        raw = pd.read_excel(path, sheet_name=sheet_name)
        code_col, name_col, year_col, dim_col, est_col = find_wgi_columns(raw)

        sub_cols = [code_col, year_col, est_col]
        if name_col is not None:
            sub_cols.append(name_col)
        if dim_col is not None:
            sub_cols.append(dim_col)

        sub = raw[sub_cols].copy()
        sub["country_code"] = sub[code_col].astype(str).str.strip()
        sub["year"] = pd.to_numeric(sub[year_col], errors="coerce").astype("Int64")
        sub[out_col] = pd.to_numeric(sub[est_col], errors="coerce")

        if name_col is not None:
            sub["country_name"] = sub[name_col].astype(str).str.strip()

        sub = sub[sub["year"].notna()].copy()
        sub["year"] = sub["year"].astype(int)
        sub = sub[(sub["year"] >= CFG.YEAR_START) & (sub["year"] <= CFG.YEAR_END)].copy()

        keep = ["country_code", "year", out_col]
        if "country_name" in sub.columns:
            keep.insert(1, "country_name")

        sub = sub[keep].drop_duplicates(subset=["country_code", "year"], keep="last")
        frames.append(sub)

        print(f"WGI sheet {sheet_name} -> {out_col}: {sub.shape}")

    if not frames:
        raise ValueError("No WGI sheets were successfully loaded")

    from functools import reduce

    def merge_two(left, right):
        on_cols = ["country_code", "year"]
        if "country_name" in left.columns and "country_name" in right.columns:
            on_cols = ["country_code", "country_name", "year"]
        elif "country_name" in right.columns and "country_name" not in left.columns:
            left = left.merge(
                right[["country_code", "year", "country_name"]].drop_duplicates(),
                on=["country_code", "year"],
                how="left"
            )
            on_cols = ["country_code", "country_name", "year"]
        elif "country_name" in left.columns and "country_name" not in right.columns:
            on_cols = ["country_code", "country_name", "year"]

        return pd.merge(left, right, on=on_cols, how="outer")

    df = reduce(merge_two, frames)

    if "country_name" not in df.columns:
        df["country_name"] = df["country_code"]

    wgi_cols = list(target_codes.values())
    present_wgi_cols = [c for c in wgi_cols if c in df.columns]

    df = df[["country_code", "country_name", "year"] + present_wgi_cols].copy()
    df["country_code"] = df["country_code"].astype(str).str.strip()
    df["country_name"] = df["country_name"].astype(str).str.strip()
    df["year"] = pd.to_numeric(df["year"], errors="coerce").astype("Int64")
    df = df[df["year"].notna()].copy()
    df["year"] = df["year"].astype(int)

    df = df.drop_duplicates(subset=["country_code", "year"], keep="last")
    df = df.sort_values(["country_code", "year"]).reset_index(drop=True)

    print(f"WGI merged shape: {df.shape}")
    print(f"WGI countries: {df['country_code'].nunique()}")
    print(f"WGI years: {df['year'].min()}-{df['year'].max()}")
    print(f"WGI dimensions present: {present_wgi_cols}")

    return df

with Timer("Local WDI/WGI load"):
    df_wdi = load_wdi_local(WDI_SOURCE)
    df_wgi = load_wgi_local(WGI_SOURCE)

df_wdi.to_csv(WDI_CACHE, index=False)
df_wgi.to_csv(WGI_CACHE, index=False)

print(f"WDI cache saved: {WDI_CACHE}")
print(f"WGI cache saved: {WGI_CACHE}")

print("Final loaded data:")
print("WDI shape:", df_wdi.shape)
print("WGI shape:", df_wgi.shape)

WDI loaded: (6076, 39)
WDI countries: 217
WDI years: 1996-2023
WDI features present: 36 / 37
WDI missing features: ['co2_pc']
WGI sheet va -> wgivoice: (5234, 4)
WGI sheet pv -> wgistability: (5214, 4)
WGI sheet ge -> wgigovteff: (5127, 4)
WGI sheet rq -> wgiregqual: (5128, 4)
WGI sheet rl -> wgirulelaw: (5261, 4)
WGI sheet cc -> wgicorruption: (5158, 4)
WGI merged shape: (5266, 9)
WGI countries: 216
WGI years: 1996-2023
WGI dimensions present: ['wgivoice', 'wgistability', 'wgigovteff', 'wgiregqual', 'wgirulelaw', 'wgicorruption']
⏱  Local WDI/WGI load: 29.16s
WDI cache saved: data/wdi_panel_raw.csv
WGI cache saved: data/wgi_panel_raw.csv
Final loaded data:
WDI shape: (6076, 39)
WGI shape: (5266, 9)


In [9]:
# CELL 5.1 SCHEMA HARMONIZATION FOR LOCAL WDI/WGI
# Insert AFTER Cell 5 and BEFORE Cell 6

from pathlib import Path
import pandas as pd

def _standardize_keys(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    rename = {}

    if "country_code" in df.columns and "countrycode" not in df.columns:
        rename["country_code"] = "countrycode"
    if "country_name" in df.columns and "countryname" not in df.columns:
        rename["country_name"] = "countryname"

    if rename:
        df = df.rename(columns=rename)

    if "countrycode" not in df.columns:
        raise ValueError(f"Missing country code column. Available: {list(df.columns)}")
    if "year" not in df.columns:
        raise ValueError(f"Missing year column. Available: {list(df.columns)}")

    df["countrycode"] = df["countrycode"].astype(str).str.strip()
    df["year"] = pd.to_numeric(df["year"], errors="coerce")
    df = df[df["year"].notna()].copy()
    df["year"] = df["year"].astype(int)

    if "countryname" not in df.columns:
        df["countryname"] = df["countrycode"]
    else:
        df["countryname"] = df["countryname"].astype(str).str.strip()

    return df


WDI_RENAME_MAP = {
    # Macroeconomics
    "gdp_growth": "gdpgrowth",
    "gdp_pc_usd": "gdppcusd",
    "gdp_pc_growth": "gdppcgrowth",
    "inflation": "inflation",
    "unemployment": "unemployment",
    "gross_capital_form": "grosscapitalform",
    "govt_consumption": "govtconsumption",

    # Trade / Finance
    "exports_gdp": "exportsgdp",
    "imports_gdp": "importsgdp",
    "trade_gdp": "tradegdp",
    "fdi_gdp": "fdigdp",
    "manufacturing_gdp": "manufacturinggdp",
    "hightech_exports": "hitechexports",

    # RD / Science
    "rd_gdp": "rdgdp",
    "researchers_pm": "researcherspm",
    "journal_articles": "journalarticles",
    "patents_residents": "patentsresidents",

    # Digital
    "internet_users": "internetusers",
    "mobile_subs": "mobilesubs",
    "broadband": "broadband",
    "secure_servers_pm": "secureserverspm",

    # Demographics / Human capital
    "life_expectancy": "lifeexpectancy",
    "infant_mortality": "infantmortality",
    "school_secondary": "schoolsecondary",
    "school_tertiary": "schooltertiary",
    "edu_gdp": "edugdp",
    "labor_participation": "laborparticipation",
    "female_labor": "femalelabor",
    "gini_index": "giniindex",
    "pop_growth": "popgrowth",
    "urban_pop_pct": "urbanpoppct",

    # Energy / Environment
    "co2_pc": "co2pc",
    "renewable_energy": "renewableenergy",
    "electricity_access": "electricityaccess",
    "energy_use_pc": "energyusepc",
    "forest_area_pct": "forestareapct",
    "nat_resources_gdp": "natresourcesgdp",
}

# Defensive aliases in case local CSV uses slightly different headers
WDI_ALIASES = {
    "gdp_per_capita_usd": "gdppcusd",
    "gdp_per_capita_growth": "gdppcgrowth",
    "gross_capital_formation": "grosscapitalform",
    "government_consumption": "govtconsumption",
    "exports_pct_gdp": "exportsgdp",
    "imports_pct_gdp": "importsgdp",
    "trade_pct_gdp": "tradegdp",
    "fdi_pct_gdp": "fdigdp",
    "manufacturing_pct_gdp": "manufacturinggdp",
    "high_tech_exports": "hitechexports",
    "rd_expenditure_gdp": "rdgdp",
    "researchers_per_million": "researcherspm",
    "mobile_subscriptions": "mobilesubs",
    "fixed_broadband": "broadband",
    "secure_servers_per_million": "secureserverspm",
    "secondary_enrollment": "schoolsecondary",
    "tertiary_enrollment": "schooltertiary",
    "education_gdp": "edugdp",
    "labor_force_participation": "laborparticipation",
    "female_labor_participation": "femalelabor",
    "population_growth": "popgrowth",
    "urban_population_pct": "urbanpoppct",
    "co2_per_capita": "co2pc",
    "renewables_pct": "renewableenergy",
    "electricity_access_pct": "electricityaccess",
    "energy_use_per_capita": "energyusepc",
    "forest_area_percent": "forestareapct",
    "natural_resources_rents_gdp": "natresourcesgdp",
}

WGI_RENAME_MAP = {
    "wgivoice": "wgivoice",
    "wgistability": "wgistability",
    "wgigovteff": "wgigovteff",
    "wgiregqual": "wgiregqual",
    "wgirulelaw": "wgirulelaw",
    "wgicorruption": "wgicorruption",
}

# 1) Standardize keys
df_wdi = _standardize_keys(df_wdi)
df_wgi = _standardize_keys(df_wgi)

# 2) Apply WDI renaming
rename_candidates = {}
for src, dst in {**WDI_RENAME_MAP, **WDI_ALIASES}.items():
    if src in df_wdi.columns and dst not in df_wdi.columns:
        rename_candidates[src] = dst

if rename_candidates:
    df_wdi = df_wdi.rename(columns=rename_candidates)

# 3) Apply WGI renaming defensively if needed
wgi_aliases = {
    "voice": "wgivoice",
    "political_stability": "wgistability",
    "government_effectiveness": "wgigovteff",
    "regulatory_quality": "wgiregqual",
    "rule_of_law": "wgirulelaw",
    "control_of_corruption": "wgicorruption",
}
wgi_rename_candidates = {}
for src, dst in {**WGI_RENAME_MAP, **wgi_aliases}.items():
    if src in df_wgi.columns and dst not in df_wgi.columns:
        wgi_rename_candidates[src] = dst

if wgi_rename_candidates:
    df_wgi = df_wgi.rename(columns=wgi_rename_candidates)

# 4) Keep all columns, but coerce known model features to numeric
for c in CFG.INDICATORS_WDI.keys():
    if c in df_wdi.columns:
        df_wdi[c] = pd.to_numeric(df_wdi[c], errors="coerce")

for c in CFG.WGI_CODES.keys():
    if c in df_wgi.columns:
        df_wgi[c] = pd.to_numeric(df_wgi[c], errors="coerce")

# 5) Remove exact duplicate country-year rows if any
df_wdi = df_wdi.sort_values(["countrycode", "year"]).drop_duplicates(
    subset=["countrycode", "year"], keep="last"
).reset_index(drop=True)

df_wgi = df_wgi.sort_values(["countrycode", "year"]).drop_duplicates(
    subset=["countrycode", "year"], keep="last"
).reset_index(drop=True)

# 6) Save harmonized caches for reproducibility
df_wdi.to_csv(Path(CFG.DATA_DIR) / "wdi_panel_harmonized.csv", index=False)
df_wgi.to_csv(Path(CFG.DATA_DIR) / "wgi_panel_harmonized.csv", index=False)

# 7) Diagnostics
wdi_present = [c for c in CFG.INDICATORS_WDI.keys() if c in df_wdi.columns]
wdi_missing = [c for c in CFG.INDICATORS_WDI.keys() if c not in df_wdi.columns]

wgi_present = [c for c in CFG.WGI_CODES.keys() if c in df_wgi.columns]
wgi_missing = [c for c in CFG.WGI_CODES.keys() if c not in df_wgi.columns]

print("Cell 5.1 done")
print("WDI shape:", df_wdi.shape)
print("WGI shape:", df_wgi.shape)
print("WDI keys ok:", {"countrycode", "countryname", "year"}.issubset(df_wdi.columns))
print("WGI keys ok:", {"countrycode", "countryname", "year"}.issubset(df_wgi.columns))
print(f"WDI mapped features: {len(wdi_present)}/{len(CFG.INDICATORS_WDI)}")
print("WDI missing:", wdi_missing)
print(f"WGI mapped features: {len(wgi_present)}/{len(CFG.WGI_CODES)}")
print("WGI missing:", wgi_missing)
print("Harmonized files saved:")
print(Path(CFG.DATA_DIR) / "wdi_panel_harmonized.csv")
print(Path(CFG.DATA_DIR) / "wgi_panel_harmonized.csv")

Cell 5.1 done
WDI shape: (6076, 39)
WGI shape: (5266, 9)
WDI keys ok: True
WGI keys ok: True
WDI mapped features: 4/37
WDI missing: ['gdp_growth', 'gdp_pc_usd', 'gdp_pc_growth', 'gross_capital_form', 'govt_consumption', 'exports_gdp', 'imports_gdp', 'trade_gdp', 'fdi_gdp', 'manufacturing_gdp', 'rd_gdp', 'researchers_pm', 'journal_articles', 'patents_residents', 'internet_users', 'mobile_subs', 'secure_servers_pm', 'life_expectancy', 'infant_mortality', 'school_secondary', 'school_tertiary', 'edu_gdp', 'labor_participation', 'female_labor', 'gini_index', 'pop_growth', 'urban_pop_pct', 'co2_pc', 'renewable_energy', 'electricity_access', 'energy_use_pc', 'forest_area_pct', 'nat_resources_gdp']
WGI mapped features: 0/6
WGI missing: ['wgi_voice', 'wgi_stability', 'wgi_govt_eff', 'wgi_reg_qual', 'wgi_rule_law', 'wgi_corruption']
Harmonized files saved:
data/wdi_panel_harmonized.csv
data/wgi_panel_harmonized.csv


In [10]:
# ─── CELL 6: DATA ENGINEERING + UNIFIED CANONICAL FEATURE SPACE ──────────

from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.preprocessing import RobustScaler

# ------------------------------------------------------------------
# 0. Canonical feature dictionaries AFTER Cell 5.1 harmonization
# ------------------------------------------------------------------
CANONICAL_WDI_FEATS = [
    # macro
    "gdpgrowth", "gdppcusd", "gdppcgrowth", "inflation", "unemployment",
    "grosscapitalform", "govtconsumption",
    # trade & finance
    "exportsgdp", "importsgdp", "tradegdp", "fdigdp", "manufacturinggdp", "hitechexports",
    # innovation
    "rdgdp", "researcherspm", "journalarticles", "patentsresidents",
    # digital
    "internetusers", "mobilesubs", "broadband", "secureserverspm",
    # human capital
    "lifeexpectancy", "infantmortality", "schoolsecondary", "schooltertiary",
    "edugdp", "laborparticipation", "femalelabor", "giniindex", "popgrowth", "urbanpoppct",
    # energy / environment
    "co2pc", "renewableenergy", "electricityaccess", "energyusepc", "forestareapct", "natresourcesgdp",
]

CANONICAL_WGI_FEATS = [
    "wgivoice", "wgistability", "wgigovteff",
    "wgiregqual", "wgirulelaw", "wgicorruption",
]

CANONICAL_ALL_FEATS = CANONICAL_WDI_FEATS + CANONICAL_WGI_FEATS

# ------------------------------------------------------------------
# 1. Key harmonization
# ------------------------------------------------------------------
def harmonize_keys(df: pd.DataFrame, name: str) -> pd.DataFrame:
    df = df.copy()

    rename_map = {}
    if "country_code" in df.columns and "countrycode" not in df.columns:
        rename_map["country_code"] = "countrycode"
    if "country_name" in df.columns and "countryname" not in df.columns:
        rename_map["country_name"] = "countryname"
    if rename_map:
        df = df.rename(columns=rename_map)

    if "countrycode" not in df.columns:
        raise ValueError(f"{name} missing countrycode")
    if "year" not in df.columns:
        raise ValueError(f"{name} missing year")

    df["countrycode"] = df["countrycode"].astype(str).str.strip()
    df["year"] = pd.to_numeric(df["year"], errors="coerce")
    df = df[df["year"].notna()].copy()
    df["year"] = df["year"].astype(int)

    if "countryname" not in df.columns:
        df["countryname"] = df["countrycode"]
    else:
        df["countryname"] = df["countryname"].astype(str).str.strip()

    return df

df_wdi = harmonize_keys(df_wdi, "WDI")
df_wgi = harmonize_keys(df_wgi, "WGI")

# ------------------------------------------------------------------
# 2. Detect actually available canonical columns
# ------------------------------------------------------------------
WDI_FEATS = [c for c in CANONICAL_WDI_FEATS if c in df_wdi.columns]
WGI_FEATS = [c for c in CANONICAL_WGI_FEATS if c in df_wgi.columns]
ALL_FEATS = WDI_FEATS + WGI_FEATS

print("Canonical WDI features present:", len(WDI_FEATS), "/", len(CANONICAL_WDI_FEATS))
print("Canonical WGI features present:", len(WGI_FEATS), "/", len(CANONICAL_WGI_FEATS))

missing_wdi = [c for c in CANONICAL_WDI_FEATS if c not in WDI_FEATS]
missing_wgi = [c for c in CANONICAL_WGI_FEATS if c not in WGI_FEATS]

print("Missing WDI canonical features:", missing_wdi)
print("Missing WGI canonical features:", missing_wgi)

if len(ALL_FEATS) < 10:
    raise ValueError(
        f"Too few canonical features available before merge: {len(ALL_FEATS)}. "
        "This means Cell 5.1 did not produce the expected harmonized columns."
    )

# ------------------------------------------------------------------
# 3. Merge WDI + WGI
# ------------------------------------------------------------------
wdi_cols = ["countrycode", "countryname", "year"] + WDI_FEATS
wgi_cols = ["countrycode", "countryname", "year"] + WGI_FEATS

panel = pd.merge(
    df_wdi[wdi_cols].copy(),
    df_wgi[wgi_cols].copy(),
    on=["countrycode", "year"],
    how="outer",
    suffixes=("_wdi", "_wgi")
)

if "countryname_wdi" in panel.columns and "countryname_wgi" in panel.columns:
    panel["countryname"] = panel["countryname_wdi"].fillna(panel["countryname_wgi"])
    panel = panel.drop(columns=["countryname_wdi", "countryname_wgi"])
elif "countryname_wdi" in panel.columns:
    panel = panel.rename(columns={"countryname_wdi": "countryname"})
elif "countryname_wgi" in panel.columns:
    panel = panel.rename(columns={"countryname_wgi": "countryname"})
elif "countryname" not in panel.columns:
    panel["countryname"] = panel["countrycode"]

panel = panel[(panel["year"] >= CFG.YEAR_START) & (panel["year"] <= CFG.YEAR_END)].copy()

for c in ALL_FEATS:
    panel[c] = pd.to_numeric(panel[c], errors="coerce")

# ------------------------------------------------------------------
# 4. Coverage filter
# ------------------------------------------------------------------
coverage = panel.groupby("countrycode")[ALL_FEATS].apply(lambda g: g.notna().mean().mean())
good_ctry = coverage[coverage >= CFG.MIN_COVERAGE].index.tolist()

if len(good_ctry) == 0:
    raise ValueError("No countries passed MIN_COVERAGE using canonical feature space")

panel = panel[panel["countrycode"].isin(good_ctry)].copy()
coverage = coverage.loc[good_ctry]

top_ctry = coverage.nlargest(min(CFG.MAX_COUNTRIES, len(coverage))).index.tolist()
panel = panel[panel["countrycode"].isin(top_ctry)].copy()

# ------------------------------------------------------------------
# 5. Winsorise + impute
# ------------------------------------------------------------------
for c in ALL_FEATS:
    if panel[c].notna().any():
        lo, hi = panel[c].quantile([0.01, 0.99])
        panel[c] = panel[c].clip(lo, hi)

panel = panel.sort_values(["countrycode", "year"]).reset_index(drop=True)

for c in ALL_FEATS:
    panel[c] = panel.groupby("countrycode")[c].transform(lambda x: x.ffill().bfill())

panel[ALL_FEATS] = panel[ALL_FEATS].fillna(panel[ALL_FEATS].median())

# ------------------------------------------------------------------
# 6. ONE maximal common feature set for all models
# ------------------------------------------------------------------
MODEL_FEATS = [c for c in CANONICAL_ALL_FEATS if c in panel.columns]

if len(MODEL_FEATS) < 10:
    raise ValueError(f"Too few model features after merge: {len(MODEL_FEATS)}")

FEAT_COLS = MODEL_FEATS.copy()
CFG.STATE_DIM = len(FEAT_COLS)

# ------------------------------------------------------------------
# 7. Normalize using train years only
# ------------------------------------------------------------------
SCALER = RobustScaler()

TRAIN_YEARS = list(range(CFG.YEAR_START, 2015))
VAL_YEARS = list(range(2015, 2019))
TEST_YEARS = list(range(2019, CFG.YEAR_END + 1))

mask_train = panel["year"].isin(TRAIN_YEARS)
SCALER.fit(panel.loc[mask_train, FEAT_COLS].values)
panel[FEAT_COLS] = SCALER.transform(panel[FEAT_COLS].values)

# ------------------------------------------------------------------
# 8. Backward-compatible aliases for downstream cells
# ------------------------------------------------------------------
panel["country_code"] = panel["countrycode"]
panel["country_name"] = panel["countryname"]

# ------------------------------------------------------------------
# 9. Save and report
# ------------------------------------------------------------------
panel.to_csv(Path(CFG.DATA_DIR) / "panel_clean.csv", index=False)

COUNTRIES = sorted(panel["countrycode"].unique().tolist())
N_AGENTS = len(COUNTRIES)

print("\nPanel shape:", panel.shape)
print("Countries:", panel["countrycode"].nunique())
print("Years:", panel["year"].min(), "-", panel["year"].max())
print("Unified feature count:", len(FEAT_COLS))
print("STATE_DIM updated to:", CFG.STATE_DIM)
print("First 15 MODEL_FEATS:", MODEL_FEATS[:15])
print("Active agents:", N_AGENTS)
print("Clean panel saved:", Path(CFG.DATA_DIR) / "panel_clean.csv")

Canonical WDI features present: 35 / 37
Canonical WGI features present: 6 / 6
Missing WDI canonical features: ['hitechexports', 'co2pc']
Missing WGI canonical features: []

Panel shape: (1400, 46)
Countries: 50
Years: 1996 - 2023
Unified feature count: 41
STATE_DIM updated to: 41
First 15 MODEL_FEATS: ['gdpgrowth', 'gdppcusd', 'gdppcgrowth', 'inflation', 'unemployment', 'grosscapitalform', 'govtconsumption', 'exportsgdp', 'importsgdp', 'tradegdp', 'fdigdp', 'manufacturinggdp', 'rdgdp', 'researcherspm', 'journalarticles']
Active agents: 50
Clean panel saved: data/panel_clean.csv


In [11]:
print(len(FEAT_COLS))
print(FEAT_COLS)
print(panel.head(2).T.head(60))

41
['gdpgrowth', 'gdppcusd', 'gdppcgrowth', 'inflation', 'unemployment', 'grosscapitalform', 'govtconsumption', 'exportsgdp', 'importsgdp', 'tradegdp', 'fdigdp', 'manufacturinggdp', 'rdgdp', 'researcherspm', 'journalarticles', 'patentsresidents', 'internetusers', 'mobilesubs', 'broadband', 'secureserverspm', 'lifeexpectancy', 'infantmortality', 'schoolsecondary', 'schooltertiary', 'edugdp', 'laborparticipation', 'femalelabor', 'giniindex', 'popgrowth', 'urbanpoppct', 'renewableenergy', 'electricityaccess', 'energyusepc', 'forestareapct', 'natresourcesgdp', 'wgivoice', 'wgistability', 'wgigovteff', 'wgiregqual', 'wgirulelaw', 'wgicorruption']
                            0          1
countrycode               ARG        ARG
year                     1996       1997
gdpgrowth            0.516913   1.150909
gdppcusd            -0.219624  -0.199685
gdppcgrowth          0.334824   0.964437
inflation            5.877124   5.877124
unemployment         1.827455   1.396091
grosscapitalform    -0

In [12]:
# ─── CELL 7: CRISIS MODULE + EDA + SAFETY SCREEN (UNIFIED 41 FEATS) ──────

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# ============================================================
# 0. Backward compatibility aliases (страна/год)
# ============================================================
if "countrycode" in panel.columns and "country_code" not in panel.columns:
    panel["country_code"] = panel["countrycode"]

if "countryname" in panel.columns and "country_name" not in panel.columns:
    panel["country_name"] = panel["countryname"]

for c in ["countrycode", "country_code"]:
    if c in panel.columns:
        panel[c] = panel[c].astype(str).str.strip()

for c in ["countryname", "country_name"]:
    if c in panel.columns:
        panel[c] = panel[c].astype(str).str.strip()

panel["year"] = pd.to_numeric(panel["year"], errors="coerce")
panel = panel[panel["year"].notna()].copy()
panel["year"] = panel["year"].astype(int)

country_key = "countrycode"

# ============================================================
# 1. Crisis taxonomy (использует те же канонические фичи)
# ============================================================
CRISIS_TYPES = {
    "financial_debt": {
        "features": ["gdpgrowth", "grosscapitalform"],
        "rules": [("lt", -1.0), ("lt", -1.0)],
        "tail": False,
    },
    "inflationary": {
        "features": ["inflation"],
        "rules": [("gt", 1.5)],
        "tail": False,
    },
    "pandemic_health": {
        "features": ["gdpgrowth", "lifeexpectancy"],
        "rules": [("lt", -1.25), ("lt", -0.5)],
        "tail": False,
    },
    "energy_commodity": {
        "features": ["natresourcesgdp", "energyusepc"],
        "rules": [("gt", 0.75), ("gt", 0.5)],
        "tail": False,
    },
    "food_security": {
        "features": ["natresourcesgdp", "giniindex"],
        "rules": [("lt", -0.5), ("gt", 1.0)],
        "tail": False,
    },
    "climate_ecological": {
        "features": ["renewableenergy", "forestareapct"],
        "rules": [("lt", -1.0), ("lt", -1.0)],
        "tail": False,
    },
    "geoeconomic_frag": {
        "features": ["tradegdp", "fdigdp"],
        "rules": [("lt", -1.0), ("lt", -1.0)],
        "tail": False,
    },
    "cyber_tech": {
        "features": ["internetusers", "secureserverspm"],
        "rules": [("lt", -1.0), ("lt", -1.0)],
        "tail": False,
    },
    "elite_crisis": {
        "features": ["giniindex", "wgistability"],
        "rules": [("gt", 1.0), ("lt", -1.0)],
        "tail": False,
    },
    "legitimacy_crisis": {
        "features": ["wgivoice", "wgigovteff"],
        "rules": [("lt", -1.0), ("lt", -1.0)],
        "tail": False,
    },
    "social_cohesion": {
        "features": ["giniindex", "unemployment"],
        "rules": [("gt", 1.0), ("gt", 1.0)],
        "tail": False,
    },
    "intergenerational": {
        "features": ["schooltertiary", "unemployment"],
        "rules": [("lt", -1.0), ("gt", 1.0)],
        "tail": False,
    },
    # tail / low-prob crises (используются только в среде, не в EDA)
    "ai_misalignment": {"tail": True, "prob": 0.005},
    "epistemic_collapse": {"tail": True, "prob": 0.005},
    "infra_breakdown": {"tail": True, "prob": 0.008},
    "tech_acceleration": {"tail": True, "prob": 0.010},
    "biosecurity": {"tail": True, "prob": 0.003},
    "space_weather": {"tail": True, "prob": 0.002},
    "resource_limits": {"tail": True, "prob": 0.010},
}

CRISIS_NAMES = list(CRISIS_TYPES.keys())
N_CRISIS = len(CRISIS_NAMES)

print("Total crisis types :", N_CRISIS)
print(" Empirical :", sum(1 for v in CRISIS_TYPES.values() if not v.get("tail")))
print(" Tail/low-prob :", sum(1 for v in CRISIS_TYPES.values() if v.get("tail")))

def _rule_hit(value, op, thr):
    if pd.isna(value):
        return False
    if op == "lt":
        return value < thr
    if op == "gt":
        return value > thr
    return False

def compute_crisis_vector(row: pd.Series) -> np.ndarray:
    vec = np.zeros(N_CRISIS, dtype=np.float32)
    for i, (_, cdef) in enumerate(CRISIS_TYPES.items()):
        if cdef.get("tail"):
            continue
        checks = []
        for feat, rule in zip(cdef.get("features", []), cdef.get("rules", [])):
            if feat in row.index:
                op, thr = rule
                checks.append(_rule_hit(row[feat], op, thr))
            else:
                checks.append(False)
        if len(checks) > 0 and all(checks):
            vec[i] = 1.0
    return vec

crisis_rows = [compute_crisis_vector(row) for _, row in panel.iterrows()]
CRISIS_MATRIX = np.stack(crisis_rows)

panel["crisis_score"] = CRISIS_MATRIX.sum(axis=1)
panel["crisis_any"] = (panel["crisis_score"] > 0).astype(int)

print("\nCrisis score stats:")
print(panel["crisis_score"].describe().round(3).to_string())

crisis_prevalence = (
    pd.Series(CRISIS_MATRIX.mean(axis=0), index=CRISIS_NAMES, name="share_triggered")
    .sort_values(ascending=False)
)

print("\nTop crisis triggers:")
print(crisis_prevalence.head(10).round(3).to_string())

# ============================================================
# 2. SAFETY SCREEN для VAR БЕЙЗЛАЙНА НА ТОМ ЖЕ 41-D ФИЧЕ
# ============================================================
BASELINE_FEATS = FEAT_COLS.copy()
print("\nBaseline safety screen uses unified feature set of size:", len(BASELINE_FEATS))

from statsmodels.tsa.api import VAR

def baseline_country_is_safe(df_c: pd.DataFrame, feats, min_train_rows=12, min_test_rows=1, maxlags=1):
    train_sub = df_c[df_c["year"].isin(TRAIN_YEARS + VAL_YEARS)].sort_values("year")[feats].dropna()
    test_sub = df_c[df_c["year"].isin(TEST_YEARS)].sort_values("year")[feats].dropna()

    if len(train_sub) < min_train_rows:
        return False, "short_train"
    if len(test_sub) < min_test_rows:
        return False, "short_test"

    nunq = train_sub.nunique(dropna=True)
    if (nunq <= 1).any():
        return False, "constant_feature"

    try:
        model = VAR(train_sub.values)
        res = model.fit(maxlags=maxlags, ic=None, trend="c")
        if res.k_ar is None or res.k_ar < 1:
            return False, "zero_lag"
        _ = res.forecast(train_sub.values[-res.k_ar:], steps=1)
        return True, "ok"
    except Exception as e:
        return False, f"fit_fail:{type(e).__name__}"

baseline_screen = []
for ccode, df_c in panel.groupby(country_key):
    ok, reason = baseline_country_is_safe(df_c.copy(), BASELINE_FEATS)
    baseline_screen.append({
        "country_code": ccode,
        "ok_for_baseline": ok,
        "reason": reason,
        "n_rows": len(df_c),
        "n_trainval": int(df_c["year"].isin(TRAIN_YEARS + VAL_YEARS).sum()),
        "n_test": int(df_c["year"].isin(TEST_YEARS).sum()),
    })

df_baseline_screen = (
    pd.DataFrame(baseline_screen)
    .sort_values(["ok_for_baseline", "country_code"], ascending=[False, True])
    .reset_index(drop=True)
)

safe_countries = df_baseline_screen.loc[df_baseline_screen["ok_for_baseline"], "country_code"].tolist()
COUNTRIES = safe_countries
N_AGENTS = len(COUNTRIES)

print("\nBaseline safety screen summary:")
print(df_baseline_screen["reason"].value_counts().to_string())
print(f"\nSafe countries for unified VAR baseline: {len(COUNTRIES)} / {df_baseline_screen.shape[0]}")

output_dir = Path(CFG.OUTPUT_DIR)
output_dir.mkdir(parents=True, exist_ok=True)
df_baseline_screen.to_csv(output_dir / "table_baseline_country_screen.csv", index=False)
crisis_prevalence.to_csv(output_dir / "table_crisis_prevalence.csv", header=True)

assert len(COUNTRIES) > 0, "No safe countries left after unified-feature safety screen"

print("COUNTRIES updated for downstream baseline & MARL:", len(COUNTRIES))
print("First safe countries:", COUNTRIES[:10])

# ============================================================
# 3. EDA Figure (Fig. 1)
# ============================================================
sns.set_style("whitegrid")

fig, axes = plt.subplots(1, 4, figsize=(18, 4.5))
fig.suptitle("EDA Panel Overview", fontsize=13, fontweight="bold")

# 1) GDP growth distribution
if "gdpgrowth" in panel.columns:
    axes[0].hist(
        panel["gdpgrowth"].dropna(),
        bins=50,
        color="#2e86ab",
        edgecolor="white",
        lw=0.3,
    )
    axes[0].axvline(0, color="crimson", lw=1.5, ls="--")
    axes[0].set_title("GDP Growth Distribution")
    axes[0].set_xlabel("Normalised value")
    axes[0].set_ylabel("Count")
else:
    axes[0].text(0.5, 0.5, "gdpgrowth not available", ha="center", va="center")
    axes[0].set_axis_off()

# 2) Coverage heatmap по тем же FEAT_COLS
sel_feats = FEAT_COLS[:20]
sel_ctry = COUNTRIES[:20] if len(COUNTRIES) > 0 else []

if len(sel_ctry) > 0 and len(sel_feats) > 0:
    cov_df = (
        panel.loc[panel[country_key].isin(sel_ctry), [country_key] + sel_feats]
        .groupby(country_key)[sel_feats]
        .apply(lambda g: g.notna().mean())
    )
else:
    cov_df = pd.DataFrame()

if cov_df.shape[0] > 0 and cov_df.shape[1] > 0:
    sns.heatmap(
        cov_df.T,
        ax=axes[1],
        cmap="YlGn",
        cbar=True,
        xticklabels=True,
        yticklabels=True,
        linewidths=0.2,
    )
    axes[1].set_title("Feature Coverage Top-20")
    axes[1].tick_params(axis="x", labelsize=5, rotation=90)
    axes[1].tick_params(axis="y", labelsize=6)
else:
    axes[1].text(0.5, 0.5, "Coverage heatmap unavailable", ha="center", va="center")
    axes[1].set_axis_off()

# 3) Crisis score over time
cs_time = panel.groupby("year")["crisis_score"].mean()
axes[2].fill_between(cs_time.index, cs_time.values, alpha=0.4, color="#e63946")
axes[2].plot(cs_time.index, cs_time.values, color="#e63946", lw=1.5)
axes[2].axvspan(2008, 2010, alpha=0.2, color="orange", label="GFC")
axes[2].axvspan(2020, 2021, alpha=0.2, color="purple", label="COVID")
axes[2].legend(fontsize=8)
axes[2].set_title("Mean Crisis Score Over Time")
axes[2].set_xlabel("Year")
axes[2].set_ylabel("Mean crisis score")

# 4) Top crisis prevalence
top_prev = crisis_prevalence.head(8).sort_values(ascending=True)
axes[3].barh(top_prev.index, top_prev.values, color="#6d597a", alpha=0.85)
axes[3].set_title("Most Frequent Crisis Triggers")
axes[3].set_xlabel("Share of country-years")
axes[3].set_ylabel("")

plt.tight_layout()
ARTS.save_fig(fig, "fig1_eda_overview")

print("Cell 7 complete — unified feature space preserved")

Total crisis types : 19
 Empirical : 12
 Tail/low-prob : 7

Crisis score stats:
count    1400.000
mean        0.346
std         0.695
min         0.000
25%         0.000
50%         0.000
75%         0.000
max         5.000

Top crisis triggers:
inflationary         0.129
energy_commodity     0.057
elite_crisis         0.042
legitimacy_crisis    0.039
financial_debt       0.026
pandemic_health      0.026
social_cohesion      0.018
intergenerational    0.009
geoeconomic_frag     0.000
food_security        0.000

Baseline safety screen uses unified feature set of size: 41

Baseline safety screen summary:
reason
constant_feature    30
ok                  20

Safe countries for unified VAR baseline: 20 / 50
COUNTRIES updated for downstream baseline & MARL: 20
First safe countries: ['CRI', 'GEO', 'HRV', 'HUN', 'IDN', 'KAZ', 'LTU', 'LVA', 'MDA', 'MEX']
   🖼  Figure saved → fig1_eda_overview.pdf
Cell 7 complete — unified feature space preserved


## 📊 Experiment 1 — Baseline Foresight (VAR + Scenario Clustering)

In [13]:
# ─── CELL 8: UNIFIED-FEATURE BASELINE VAR (41D) ──────────────────────────
# Baseline использует РОВНО тот же FEAT_COLS, что и MARL/NARL.

from typing import List, Tuple
import warnings
import numpy as np
import pandas as pd

from statsmodels.tsa.api import VAR
from statsmodels.tools.sm_exceptions import ConvergenceWarning
from sklearn.metrics import mean_absolute_error, mean_squared_error

warnings.filterwarnings("ignore", category=ConvergenceWarning)
warnings.filterwarnings("ignore", message=".*No frequency information was provided.*")
warnings.filterwarnings("ignore", message=".*unsupported index.*")

BASELINE_FEATS = FEAT_COLS.copy()
FORECAST_HORIZON = 7  # лет вперёд (чуть длиннее горизонта по сравнению с 5)

print("Cell 8 starting")
print("Unified baseline feature count:", len(BASELINE_FEATS))
print("Countries passed from Cell 7:", len(COUNTRIES))

if len(BASELINE_FEATS) < 2:
    raise ValueError("Cell 8 requires at least 2 features")
if len(COUNTRIES) == 0:
    raise ValueError("COUNTRIES is empty after Cell 7")

country_key = "country_code" if "country_code" in panel.columns else "countrycode"

# ------------------------------------------------------------------
# 1. Функции для VAR и сценариев
# ------------------------------------------------------------------
def fit_var_country(
    df_c: pd.DataFrame,
    feats: List[str],
    maxlags: int = 2,           # было 1, делаем VAR(2) для чуть более богатой динамики
) -> Tuple[object, pd.DataFrame]:
    """Подбор VAR для одной страны на полном наборе фич."""
    sub = df_c.sort_values("year")[feats].dropna()
    # Требуем чуть более длинный ряд: maxlags + 5 наблюдений минимум
    min_len = maxlags + 5
    if len(sub) < min_len:
        return None, None
    try:
        model = VAR(sub.values)
        res = model.fit(maxlags=maxlags, ic=None, trend="c")
        if res.k_ar is None or res.k_ar < 1:
            return None, None
        return res, sub
    except Exception:
        return None, None


def generate_scenarios(
    forecast_mean: np.ndarray,
    cov: np.ndarray,
    nscenarios: int = 700,    # было 500, немного повышаем размер ансамбля
    seed: int = 42
) -> np.ndarray:
    """Сэмплируем сценарии из многомерной нормали вокруг VAR‑прогноза."""
    rng = np.random.default_rng(seed)
    T, D = forecast_mean.shape
    cov = np.asarray(cov, dtype=float)

    if cov.ndim == 0:
        cov = np.eye(D) * float(cov)
    if cov.shape != (D, D):
        cov = np.eye(D) * 0.05

    cov = cov + np.eye(D) * 1e-6

    return np.stack([
        forecast_mean + rng.multivariate_normal(np.zeros(D), cov, size=T)
        for _ in range(nscenarios)
    ])


LOGBASE = ExperimentLogger("baseline")

baseline_preds = []
baseline_actuals = []
baseline_countries = []
baseline_feature_sets = []
country_scenario_draws = []

# ------------------------------------------------------------------
# 2. Обучение VAR по странам
# ------------------------------------------------------------------
with Timer("Baseline — unified VAR fitting"):
    for ccode in tqdm(COUNTRIES, desc="VAR per country (41D)", ncols=80):
        df_c = panel[panel[country_key] == ccode].copy()

        mask_tr = df_c["year"].isin(TRAIN_YEARS + VAL_YEARS)
        mask_te = df_c["year"].isin(TEST_YEARS)

        train_df = df_c.loc[mask_tr].sort_values("year")
        test_df = df_c.loc[mask_te].sort_values("year")

        # На этом этапе FEAT_COLS уже полностью иммутированы и без NaN,
        # но проверка на всякий случай.
        if train_df[BASELINE_FEATS].isna().any().any():
            LOGBASE.log(country=ccode, status="skip_nan_train")
            continue
        if test_df[BASELINE_FEATS].isna().any().any():
            LOGBASE.log(country=ccode, status="skip_nan_test")
            continue

        res, sub = fit_var_country(train_df, BASELINE_FEATS, maxlags=2)
        if res is None:
            LOGBASE.log(country=ccode, status="fit_failed")
            continue

        try:
            fc = res.forecast(sub.values[-res.k_ar:], steps=FORECAST_HORIZON)
        except Exception:
            LOGBASE.log(country=ccode, status="forecast_failed")
            continue

        actual = test_df[BASELINE_FEATS].values[:FORECAST_HORIZON]
        if actual.shape[0] < 1:
            LOGBASE.log(country=ccode, status="short_test")
            continue

        min_len = min(fc.shape[0], actual.shape[0])
        fc = fc[:min_len]
        actual = actual[:min_len]

        # Оценка ковариации остатков
        try:
            sigma_u = np.asarray(res.sigma_u, dtype=float)
            if sigma_u.shape != (len(BASELINE_FEATS), len(BASELINE_FEATS)):
                sigma_u = np.eye(len(BASELINE_FEATS)) * 0.05
        except Exception:
            sigma_u = np.eye(len(BASELINE_FEATS)) * 0.05

        scen = generate_scenarios(fc, sigma_u, nscenarios=700, seed=CFG.SEED)

        baseline_preds.append(fc)
        baseline_actuals.append(actual)
        baseline_countries.append(ccode)
        baseline_feature_sets.append(BASELINE_FEATS.copy())

        country_scenario_draws.append({
            "country": ccode,
            "features": BASELINE_FEATS.copy(),
            "forecast_mean": fc,
            "forecast_scenarios": scen,
        })

        mae = mean_absolute_error(actual.ravel(), fc.ravel())
        rmse = np.sqrt(mean_squared_error(actual.ravel(), fc.ravel()))

        LOGBASE.log(
            country=ccode,
            status="ok",
            mae=float(mae),
            rmse=float(rmse),
            n_features=len(BASELINE_FEATS),
            horizon=min_len,
        )

LOGBASE.save()
df_log_base = LOGBASE.to_df()

if len(df_log_base) > 0:
    ok_mask = df_log_base["status"] == "ok"
    n_ok = int(ok_mask.sum())
else:
    ok_mask = []
    n_ok = 0

print("\nBaseline fitted countries:", n_ok, "/", len(COUNTRIES))
if len(df_log_base):
    print("\nBaseline fit status:")
    print(df_log_base["status"].value_counts().to_string())

if n_ok > 0:
    print(f"\nBaseline Results ({n_ok} countries):")
    print(
        f" MAE (mean ± std) : "
        f"{df_log_base.loc[ok_mask, 'mae'].mean():.4f} ± "
        f"{df_log_base.loc[ok_mask, 'mae'].std():.4f}"
    )
    print(
        f" RMSE (mean ± std): "
        f"{df_log_base.loc[ok_mask, 'rmse'].mean():.4f} ± "
        f"{df_log_base.loc[ok_mask, 'rmse'].std():.4f}"
    )
    print(
        f" Mean #features used: "
        f"{df_log_base.loc[ok_mask, 'n_features'].mean():.2f}"
    )

print("Baseline forecast objects ready for Cell 9")
print("Example countries:", baseline_countries[:10])

Cell 8 starting
Unified baseline feature count: 41
Countries passed from Cell 7: 20


VAR per country (41D):   0%|                             | 0/20 [00:00<?, ?it/s]

⏱  Baseline — unified VAR fitting: 20.98s
   📋 Log saved → output/log_baseline.csv

Baseline fitted countries: 20 / 20

Baseline fit status:
status
ok    20

Baseline Results (20 countries):
 MAE (mean ± std) : 5.3532 ± 8.1710
 RMSE (mean ± std): 29.2661 ± 47.1935
 Mean #features used: 41.00
Baseline forecast objects ready for Cell 9
Example countries: ['CRI', 'GEO', 'HRV', 'HUN', 'IDN', 'KAZ', 'LTU', 'LVA', 'MDA', 'MEX']


In [14]:
# ─── CELL 9: BASELINE SCENARIOS (41D PCA + KMEANS) ───────────────────────
# Использует:
#   - country_scenario_draws из Cell 8 (features == FEAT_COLS)
#   - baseline_preds, baseline_countries
# Строит:
#   - глобальные сценарии в том же 41D пространстве
#   - Figure 2 + таблицу country→scenario

from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

print("Cell 9 starting")

# sanity-check
if len(country_scenario_draws) == 0:
    raise ValueError("country_scenario_draws is empty — Cell 8 must run before Cell 9")

# --------------------------------------------------------------
# 1. Embedding стран в едином 41D пространстве
# --------------------------------------------------------------
# Для каждой страны берём средний прогноз по горизонту (FORECAST_HORIZON)
# и получаем вектор в R^41.
per_country_means = []
per_country_ids = []

for obj in country_scenario_draws:
    ccode = obj["country"]
    fc_mean = np.asarray(obj["forecast_mean"], dtype=float)  # (T, D=41)
    feats   = obj["features"]

    if list(feats) != list(FEAT_COLS):
        raise ValueError(f"Feature mismatch for {ccode}: expected FEAT_COLS, got {feats}")

    # усредняем по времени → один вектор длины 41 на страну
    vec = fc_mean.mean(axis=0)
    per_country_means.append(vec)
    per_country_ids.append(ccode)

all_fc_means = np.vstack(per_country_means)  # (N_countries, 41)
n_countries, D = all_fc_means.shape

print("Scenario embedding matrix shape:", all_fc_means.shape)

# --------------------------------------------------------------
# 2. PCA до 2D для визуализации
# --------------------------------------------------------------
n_components = min(2, D)
pca = PCA(n_components=n_components, random_state=CFG.SEED)
fc_pca = pca.fit_transform(all_fc_means)

expl_var = pca.explained_variance_ratio_
print(f"PCA explained variance: {expl_var.sum():.3f} "
      f"(PC1={expl_var[0]:.3f}, PC2={expl_var[1] if n_components > 1 else 0:.3f})")

# --------------------------------------------------------------
# 3. KMeans-кластеризация (глобальные сценарии)
# --------------------------------------------------------------
NSCENARIOS = 5
km = KMeans(
    n_clusters=NSCENARIOS,
    random_state=CFG.SEED,
    n_init=20
)
cluster_labels = km.fit_predict(fc_pca)

SCENARIO_NAMES = [
    "Sustainable Growth",
    "Stagnation",
    "Crisis Rebound",
    "Polarised Development",
    "Ecological Transition",
]

cluster_counts = np.bincount(cluster_labels, minlength=NSCENARIOS)
scenarioprobs = cluster_counts / cluster_counts.sum()

print("\nScenario Probabilities (baseline 41D):")
for i, (name, p) in enumerate(zip(SCENARIO_NAMES, scenarioprobs)):
    bar = int(p * 30) * "▮"
    print(f"  {name:28s} {p:0.3f} {bar}")

# Таблица страна → сценарий
df_scen = pd.DataFrame({
    "country": per_country_ids,
    "cluster": cluster_labels,
    "scenario_name": [SCENARIO_NAMES[i] for i in cluster_labels],
})

# --------------------------------------------------------------
# 4. Figure 2 — PCA scatter + scenario probabilities
# --------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
COLORS = ["#2e86ab", "#a23b72", "#f18f01", "#c73e1d", "#3b1f2b"]

# Left: PCA scatter стран по сценариям
for i in range(NSCENARIOS):
    mask = cluster_labels == i
    if not np.any(mask):
        continue
    axes[0].scatter(
        fc_pca[mask, 0],
        fc_pca[mask, 1] if n_components > 1 else np.zeros(mask.sum()),
        label=SCENARIO_NAMES[i],
        color=COLORS[i],
        alpha=0.8,
        s=40,
        edgecolor="white",
        linewidth=0.4,
    )

for idx, ccode in enumerate(per_country_ids):
    axes[0].text(
        fc_pca[idx, 0],
        fc_pca[idx, 1] if n_components > 1 else 0.0,
        ccode,
        fontsize=6,
        ha="center",
        va="center",
        color="black",
        alpha=0.8,
    )

axes[0].set_title("Baseline Scenario Clusters (41D → PCA)")
axes[0].set_xlabel("PC1")
if n_components > 1:
    axes[0].set_ylabel("PC2")
else:
    axes[0].set_ylabel("")
axes[0].legend(fontsize=7, loc="best")

# Right: вероятности сценариев
axes[1].bar(
    np.arange(NSCENARIOS),
    scenarioprobs,
    color=COLORS,
    alpha=0.9,
    edgecolor="white",
)
axes[1].set_xticks(np.arange(NSCENARIOS))
axes[1].set_xticklabels(
    [n.replace(" ", "\n") for n in SCENARIO_NAMES],
    fontsize=8,
)
axes[1].set_ylabel("Scenario probability")
axes[1].set_title("Baseline Scenario Probability Distribution")

plt.tight_layout()
ARTS.save_fig(fig, "fig2_baseline_scenarios_41d")

# --------------------------------------------------------------
# 5. Сохранение таблицы
# --------------------------------------------------------------
output_dir = Path(CFG.OUTPUT_DIR)
output_dir.mkdir(parents=True, exist_ok=True)
df_scen.to_csv(output_dir / "table_baseline_country_scenarios.csv", index=False)

print(f"\nScenario table saved → {output_dir / 'table_baseline_country_scenarios.csv'}")
print("Cell 9 finished (41D baseline scenarios)")

Cell 9 starting
Scenario embedding matrix shape: (20, 41)
PCA explained variance: 0.978 (PC1=0.860, PC2=0.118)

Scenario Probabilities (baseline 41D):
  Sustainable Growth           0.100 ▮▮▮
  Stagnation                   0.100 ▮▮▮
  Crisis Rebound               0.650 ▮▮▮▮▮▮▮▮▮▮▮▮▮▮▮▮▮▮▮
  Polarised Development        0.100 ▮▮▮
  Ecological Transition        0.050 ▮
   🖼  Figure saved → fig2_baseline_scenarios_41d.pdf

Scenario table saved → output/table_baseline_country_scenarios.csv
Cell 9 finished (41D baseline scenarios)


In [15]:
# ─── CELL 9.1: SCENARIO PROFILES (FOR PAPER) ─────────────────────────────
# Требует выполнение Cell 9:
#  - df_scen, SCENARIO_NAMES, cluster_labels
#  - all_fc_means (N_countries, 41)
#  - per_country_ids, FEAT_COLS

from pathlib import Path
import numpy as np
import pandas as pd

print("Cell 9.1 starting")

required_globals = ["df_scen", "SCENARIO_NAMES", "cluster_labels",
                    "all_fc_means", "per_country_ids", "FEAT_COLS"]
for name in required_globals:
    if name not in globals():
        raise RuntimeError(f"{name} not found — run Cell 9 first")

N_SCEN = len(SCENARIO_NAMES)
N_COUNTRIES, D = all_fc_means.shape

print(f"All-forecast means shape: {all_fc_means.shape}")
print(f"Scenarios: {N_SCEN}, Countries: {N_COUNTRIES}, Dim: {D}")

# ------------------------------------------------------------------
# 1. Осевые группы признаков для интерпретации сценариев
# ------------------------------------------------------------------
GROUPS = {
    "growth": [
        "gdpgrowth", "gdppcgrowth", "grosscapitalform",
        "exportsgdp", "importsgdp", "tradegdp",
        "fdigdp", "manufacturinggdp",
    ],
    "inflation_unemp": [
        "inflation", "unemployment",
    ],
    "inequality_social": [
        "giniindex", "laborparticipation", "femalelabor",
        "schoolsecondary", "schooltertiary", "edugdp",
    ],
    "digital": [
        "internetusers", "mobilesubs", "broadband", "secureserverspm",
    ],
    "green_energy": [
        "renewableenergy", "electricityaccess",
        "energyusepc", "forestareapct", "natresourcesgdp",
    ],
    "governance": [
        "wgivoice", "wgistability", "wgigovteff",
        "wgiregqual", "wgirulelaw", "wgicorruption",
    ],
}

# оставляем только реально доступные признаки
GROUPS = {
    g: [f for f in feats if f in FEAT_COLS]
    for g, feats in GROUPS.items()
}

# ------------------------------------------------------------------
# 2. Средние профили по сценариям (z-скоры в 41D)
# ------------------------------------------------------------------
rows = []

for scen_idx, scen_name in enumerate(SCENARIO_NAMES):
    mask = cluster_labels == scen_idx
    if not np.any(mask):
        continue

    scen_means = all_fc_means[mask]  # (n_scen_countries, D)

    row = {
        "scenario": scen_name,
        "n_countries": int(mask.sum()),
        "share_countries": float(mask.mean()),
    }

    for gname, feats in GROUPS.items():
        if len(feats) == 0:
            continue
        idx = [FEAT_COLS.index(f) for f in feats]
        val = scen_means[:, idx].mean()
        row[gname] = float(val)

    rows.append(row)

df_profiles = pd.DataFrame(rows)
df_profiles = df_profiles.sort_values("share_countries", ascending=False).reset_index(drop=True)

# ------------------------------------------------------------------
# 3. Сохранение таблицы профилей для статьи
# ------------------------------------------------------------------
output_dir = Path(CFG.OUTPUT_DIR)
output_dir.mkdir(parents=True, exist_ok=True)

profiles_path = output_dir / "table_baseline_scenario_profiles.csv"
df_profiles.to_csv(profiles_path, index=False)

print("Scenario profile table saved →", profiles_path)
print("Cell 9.1 finished — scenario profiles (z-scores) ready for paper")
print(df_profiles)

Cell 9.1 starting
All-forecast means shape: (20, 41)
Scenarios: 5, Countries: 20, Dim: 41
Scenario profile table saved → output/table_baseline_scenario_profiles.csv
Cell 9.1 finished — scenario profiles (z-scores) ready for paper
                scenario  n_countries  share_countries     growth  \
0         Crisis Rebound           13             0.65   0.252394   
1     Sustainable Growth            2             0.10   2.486231   
2             Stagnation            2             0.10   9.449056   
3  Polarised Development            2             0.10  10.874472   
4  Ecological Transition            1             0.05 -42.630467   

   inflation_unemp  inequality_social     digital  green_energy  governance  
0        -0.750465           0.168834    1.512701      0.301060   -0.524808  
1        -0.743326          -0.219486   24.467223      0.448251   -0.036735  
2        -8.879068           1.764595  121.336199      1.099161   -2.882221  
3        13.395851           2.663372   74.

In [16]:
# ─── CELL 10: MULTI-COUNTRY MARL ENVIRONMENT ─────────────────────────────
# Implements Dec-POMDP / Markov Game for strategic foresight

# State S_t : panel of WDI+WGI indicators per country (normalised FEAT_COLS)
# Obs o_t^i: local country state + global summary (partial observability)
# Action a_t^i: 8‑dim continuous portfolio [-1,1] (tanh output)
# Reward R_t^i: weighted composite of growth, social, tech, sustainability
# Shock K_t : stochastic crisis injection (empirical + tail risks)

class WorldModel(nn.Module):
    """Learned transition dynamics: S_{t+1} = f_θ(S_t, A_t) + ε_t"""
    def __init__(self, state_dim: int, action_dim: int, hidden: int = 128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim + action_dim, hidden),
            nn.LayerNorm(hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.ReLU(),
            nn.Linear(hidden, state_dim),
        )
        # гетероскедастический шум (однаковый по фичам, но обучаемый масштаб)
        self.log_var = nn.Parameter(torch.zeros(state_dim) - 2.0)

    def forward(
        self, state: torch.Tensor, action: torch.Tensor
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        x = torch.cat([state, action], dim=-1)
        mean = state + self.net(x)  # residual connection
        std = torch.exp(0.5 * self.log_var).clamp(1e-4, 1.0)
        return mean, std


class GlobalForesightEnv:
    """
    Multi-country MARL environment for strategic foresight.
    Dec-POMDP / CTDE:
    - Actors видят локальное состояние + глобальные агрегаты.
    - Central critic видит глобальный state + все действия.

    Использует:
    - panel (train+val сплит для обучения world model),
    - FEAT_COLS как список нормализованных признаков.
    """
    def __init__(self, panel: pd.DataFrame, feat_cols: List[str], cfg: Config):
        self.cfg = cfg

        # используем только те фичи, которые реально есть в панели
        self.feat_cols: List[str] = [f for f in feat_cols if f in panel.columns]
        self.state_dim: int = len(self.feat_cols)
        self.act_dim: int = cfg.ACTION_DIM

        # список стран-agents (ключ country_code уже приведён в Cell 7)
        self.countries: List[str] = sorted(panel.country_code.unique().tolist())
        self.n_agents: int = len(self.countries)

        # временная сетка
        years = sorted(panel.year.unique())
        self.years: List[int] = years
        self.n_years: int = len(years)
        self.year2idx: Dict[int, int] = {y: i for i, y in enumerate(years)}

        # исторический тензор (C, T, D)
        data = np.zeros((self.n_agents, self.n_years, self.state_dim), dtype=np.float32)
        for ci, cc in enumerate(self.countries):
            sub = panel[panel.country_code == cc].sort_values("year")
            for _, row in sub.iterrows():
                yi = self.year2idx.get(row.year)
                if yi is not None:
                    data[ci, yi] = [row.get(f, 0.0) for f in self.feat_cols]
        self.data = data  # (C, T, D)

        # world model (динамика переходов)
        self.world_model = WorldModel(self.state_dim, self.act_dim).to(DEVICE)
        self._pretrain_world_model()

        # векторы шоков кризисов
        self._build_crisis_vectors()

        # состояние эпизода
        self.t: int = 0
        self.states: Optional[np.ndarray] = None  # (C, D)
        self._last_state_summary: Optional[Dict[str, float]] = None

    # ------------------------------------------------------------------ #
    # World model pretraining on historical transitions
    # ------------------------------------------------------------------ #
    def _pretrain_world_model(self, n_steps: int = 800) -> None:
        """Быстрый pre-train world model на исторических переходах (no actions)."""
        opt = torch.optim.Adam(self.world_model.parameters(), lr=1e-3)
        rng = np.random.default_rng(self.cfg.SEED + 42)

        # если по каким‑то причинам мало лет — выходим сразу
        if self.n_years < 2 or self.n_agents == 0:
            return

        for _ in range(n_steps):
            ci = rng.integers(0, self.n_agents)
            ti = rng.integers(0, self.n_years - 1)

            s = torch.tensor(
                self.data[ci, ti], dtype=torch.float32, device=DEVICE
            ).unsqueeze(0)
            sn = torch.tensor(
                self.data[ci, ti + 1], dtype=torch.float32, device=DEVICE
            ).unsqueeze(0)
            a = torch.zeros(1, self.act_dim, device=DEVICE)

            mu, sig = self.world_model(s, a)
            loss = F.mse_loss(mu, sn)

            opt.zero_grad()
            loss.backward()
            opt.step()

    # ------------------------------------------------------------------ #
    # Crisis shock vectors (effects in state space)
    # ------------------------------------------------------------------ #
    def _build_crisis_vectors(self) -> None:
        """Pre-compute per-crisis effect vectors (подписи в координатах state)."""
        rng = np.random.default_rng(self.cfg.SEED + 7)
        self.crisis_effects: Dict[str, np.ndarray] = {}

        # эмпирические кризисы — умеренные шоки
        emp_crises = [k for k, v in CRISIS_TYPES.items() if not v.get("tail")]
        for cname in emp_crises:
            eff = rng.normal(0, 0.25, self.state_dim).astype(np.float32)
            # сильнее бьём по росту
            for feat in ["gdpgrowth", "gdppcgrowth"]:
                if feat in self.feat_cols:
                    idx = self.feat_cols.index(feat)
                    eff[idx] = -abs(eff[idx]) * 2.0
            self.crisis_effects[cname] = eff

        # хвостовые кризисы — большие шоки (немного смягчены)
        for cname, cdef in CRISIS_TYPES.items():
            if cdef.get("tail"):
                eff = rng.normal(0, 0.7, self.state_dim).astype(np.float32)
                self.crisis_effects[cname] = eff

    # ------------------------------------------------------------------ #
    # State summary helpers
    # ------------------------------------------------------------------ #
    def _build_state_summary(self, states: Optional[np.ndarray] = None) -> Dict[str, float]:
        """
        Агрегированное summary текущего состояния среды.
        Возвращает backward-compatible ключи для Cell 13 / 14:
        gdp_growth, rd_gdp, wgi_govt_eff, co2_pc, renewable_energy и т.д.
        """
        if states is None:
            states = self.states

        if states is None:
            return {
                "gdp_growth": 0.0,
                "gdp_pc_usd": 0.0,
                "rd_gdp": 0.0,
                "inflation": 0.0,
                "unemployment": 0.0,
                "gdp_growth_vol3": 0.0,
                "gini_index": 0.0,
                "school_tertiary": 0.0,
                "crisis_score": 0.0,
                "reserves": 0.0,
                "wgi_govt_eff": 0.0,
                "wgi_rule_law": 0.0,
                "wgi_corruption": 0.0,
                "co2_pc": 0.0,
                "renewable_energy": 0.0,
                "forest_area_pct": 0.0,
                "energy_use_pc": 0.0,
                "action_entropy": 0.0,
                "ai_switches": 0.0,
            }

        feat = self.feat_cols

        def _get_mean(name: str) -> float:
            if name in feat:
                return float(np.nanmean(states[:, feat.index(name)]))
            return 0.0

        summary = {
            # backward-compatible names expected by Cell 13
            "gdp_growth": _get_mean("gdpgrowth"),
            "gdp_pc_usd": _get_mean("gdppcusd"),
            "rd_gdp": _get_mean("rdgdp"),
            "inflation": _get_mean("inflation"),
            "unemployment": _get_mean("unemployment"),
            "gini_index": _get_mean("giniindex"),
            "school_tertiary": _get_mean("schooltertiary"),
            "wgi_govt_eff": _get_mean("wgigovteff"),
            "wgi_rule_law": _get_mean("wgirulelaw"),
            "wgi_corruption": _get_mean("wgicorruption"),
            "co2_pc": _get_mean("co2pc"),
            "renewable_energy": _get_mean("renewableenergy"),
            "forest_area_pct": _get_mean("forestareapct"),
            "energy_use_pc": _get_mean("energyusepc"),

            # дополнительные сигналы
            "crisis_score": 0.0,
            "reserves": 0.0,
            "action_entropy": 0.0,
            "ai_switches": 0.0,
        }

        if "gdpgrowth" in feat:
            g = states[:, feat.index("gdpgrowth")]
            summary["gdp_growth_vol3"] = float(np.nanstd(g))
        else:
            summary["gdp_growth_vol3"] = 0.0

        return summary

    def last_state_summary(self) -> Dict[str, float]:
        if self._last_state_summary is None:
            self._last_state_summary = self._build_state_summary()
        return self._last_state_summary

    # ------------------------------------------------------------------ #
    # Reset / Observations
    # ------------------------------------------------------------------ #
    def reset(self, start_year: Optional[int] = None) -> Tuple[np.ndarray, np.ndarray]:
        """Reset к случайному году в train‑сплите (или заданному)."""
        rng = np.random.default_rng()
        if start_year is None:
            valid_years = [y for y in self.years if y in TRAIN_YEARS]
            if len(valid_years) == 0:
                yr = self.years[0]
            else:
                yr = rng.choice(valid_years)
        else:
            yr = start_year

        yi = self.year2idx.get(yr, 0)
        self.states = self.data[:, yi, :].copy()  # (C, D)
        self.t = 0
        self._last_state_summary = self._build_state_summary(self.states)

        obs = self._get_obs()
        global_state = self.states.flatten()
        return obs, global_state

    def _get_obs(self) -> np.ndarray:
        """Локальное наблюдение: own state + global mean + global std."""

        # Если среда ещё не reset'нута — возвращаем dummy‑наблюдение нужной размерности.
        if self.states is None:
            dummy = np.zeros((self.n_agents, 3 * self.state_dim), dtype=np.float32)
            return dummy

        global_mean = self.states.mean(axis=0)  # (D,)
        global_std = self.states.std(axis=0)    # (D,)

        obs = np.concatenate(
            [
                self.states,                             # (C, D)
                np.tile(global_mean, (self.n_agents, 1)),# (C, D)
                np.tile(global_std, (self.n_agents, 1)), # (C, D)
            ],
            axis=1,
        )  # → (C, 3D)

        return obs.astype(np.float32)

    # ------------------------------------------------------------------ #
    # Crisis sampling
    # ------------------------------------------------------------------ #
    def _sample_crisis(self) -> Optional[str]:
        rng = np.random.default_rng()
        # базовая эмпирическая вероятность кризиса
        if rng.random() < self.cfg.CRISIS_INJECTION_PROB:
            emp = [k for k, v in CRISIS_TYPES.items() if not v.get("tail")]
            return rng.choice(emp)

        # хвостовые риски (по индивидуальным prob)
        for cname, cdef in CRISIS_TYPES.items():
            if cdef.get("tail") and rng.random() < cdef.get("prob", 0.0):
                return cname
        return None

    # ------------------------------------------------------------------ #
    # Step
    # ------------------------------------------------------------------ #
    def step(
        self, actions: np.ndarray
    ) -> Tuple[np.ndarray, np.ndarray, np.ndarray, bool, dict]:
        """
        actions: (C, A) в R, мы сами применяем tanh.
        returns:
        obs (C, 3D),
        rewards (C,),
        global_s (C*D,),
        done bool,
        info dict
        """
        self.t += 1

        # ---- world model transition -----------------------------------
        with torch.no_grad():
            s_t = torch.tensor(self.states, dtype=torch.float32, device=DEVICE)
            a_t = torch.tensor(actions, dtype=torch.float32, device=DEVICE)
            a_t = torch.tanh(a_t)  # squash в [-1,1]

            mu, sig = self.world_model(s_t, a_t)
            noise = torch.randn_like(mu) * sig * 0.15  # было 0.2, чуть снижаем шум
            s_next = (mu + noise).cpu().numpy()

        # ---- crisis shock ---------------------------------------------
        crisis = self._sample_crisis()
        info = {"crisis": crisis}
        if crisis and crisis in self.crisis_effects:
            eff = self.crisis_effects[crisis]
            rng = np.random.default_rng()
            hit = rng.choice(
                self.n_agents, size=max(1, self.n_agents // 3), replace=False
            )
            s_next[hit] += eff * rng.uniform(0.5, 1.5)

        # clip state в разумные границы z‑пространства
        self.states = np.clip(s_next, -5.0, 5.0)

        # ---- reward ----------------------------------------------------
        rewards = self._compute_rewards(self.states)

        self._last_state_summary = self._build_state_summary(self.states)
        if crisis is not None:
            self._last_state_summary["crisis_score"] = 1.0
        info["state_summary"] = self._last_state_summary

        obs = self._get_obs()
        global_state = self.states.flatten()
        done = self.t >= self.cfg.EPISODE_LEN

        return obs, rewards, global_state, done, info

    # ------------------------------------------------------------------ #
    # Reward function
    # ------------------------------------------------------------------ #
    def _compute_rewards(self, states: np.ndarray) -> np.ndarray:
        """
        R_i = λ1*G + λ2*S + λ3*H + λ4*T + λ5*E - λ6*V - λ7*C
        Все компоненты берём в z‑пространстве (после RobustScaler).
        """
        rew = np.zeros(self.n_agents, dtype=np.float32)
        feat = self.feat_cols

        def _get(name: str) -> np.ndarray:
            return states[:, feat.index(name)] if name in feat else np.zeros(self.n_agents)

        # G — экономический рост
        rew += 0.25 * _get("gdpgrowth")
        rew += 0.10 * _get("gdppcgrowth")

        # S — социальная стабильность
        rew -= 0.10 * _get("unemployment")
        rew -= 0.05 * _get("giniindex")

        # H — человеческий капитал
        rew += 0.10 * _get("lifeexpectancy")
        rew += 0.05 * _get("schooltertiary")

        # T — технология и цифровизация
        rew += 0.15 * _get("rdgdp")
        rew += 0.05 * _get("internetusers")

        # E — устойчивость / зелёный переход
        rew += 0.10 * _get("renewableenergy")
        rew -= 0.05 * _get("co2pc")

        # V — волатильность (инфляция и др.)
        rew -= 0.05 * np.abs(_get("inflation"))

        # C — системная нестабильность (дисперсия по всем координатам state)
        rew -= 0.15 * states.std(axis=1)

        return rew.astype(np.float32)


# ── Instantiate environment with current panel / FEAT_COLS ────────────────

ENV = GlobalForesightEnv(
    panel=panel[panel.year.isin(TRAIN_YEARS + VAL_YEARS)],
    feat_cols=FEAT_COLS,
    cfg=CFG,
)

# Чтобы корректно определить OBS_DIM, сначала делаем один reset
obs0, state0 = ENV.reset()
OBS_DIM = obs0.shape[1]  # 3 * state_dim

print("✅ Environment ready (updated for 41D state)")
print(f" Agents    : {ENV.n_agents}")
print(f" State dim : {ENV.state_dim} (|FEAT_COLS|)")
print(f" Obs dim   : {OBS_DIM} = 3 * state_dim")
print(f" Action dim: {ENV.act_dim}")
print(f" Episode len: {CFG.EPISODE_LEN} steps")

✅ Environment ready (updated for 41D state)
 Agents    : 50
 State dim : 41 (|FEAT_COLS|)
 Obs dim   : 123 = 3 * state_dim
 Action dim: 8
 Episode len: 10 steps


## 🤖 Experiment 2 — MARL (CTDE-MAPPO Multi-Country Foresight)

In [17]:
# ─── CELL 11: ACTOR-CRITIC NETWORKS (CTDE-MAPPO) ─────────────────────────
# Uses:
#   ENV      — GlobalForesightEnv (Cell 10), already instantiated
#   OBS_DIM  — per-agent obs dim (3 * state_dim)
#   CFG      — global config (learning rates, hidden size, etc.)

from torch.distributions import Normal

class CountryActor(nn.Module):
    """
    Децентрализованный актор π_i(a_i | o_i; θ_i)
    Архитектура: GRU-энкодер → MLP → гауссовская голова действий.
    Работает на локальном наблюдении агента (obs_dim).
    """
    def __init__(self, obs_dim: int, action_dim: int,
                 hidden: int = 256, dropout: float = 0.10):
        super().__init__()
        self.gru = nn.GRU(
            input_size=obs_dim,
            hidden_size=hidden,
            num_layers=1,
            batch_first=True,
            dropout=0.0,
        )
        self.mlp = nn.Sequential(
            nn.LayerNorm(hidden),
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, hidden // 2),
            nn.ReLU(),
        )
        self.mean_head = nn.Linear(hidden // 2, action_dim)
        self.log_std = nn.Parameter(torch.zeros(action_dim))

        nn.init.orthogonal_(self.mean_head.weight, gain=0.01)

    def _encode(self, obs: torch.Tensor) -> torch.Tensor:
        """
        obs: (B, D) или (B, T, D).
        Возвращает эмбеддинг последнего шага: (B, H/2).
        """
        if obs.dim() == 2:
            obs = obs.unsqueeze(1)  # (B, 1, D)
        out, _ = self.gru(obs)      # (B, T, H)
        h = out[:, -1]              # (B, H)
        return self.mlp(h)          # (B, H/2)

    def forward(self, obs: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        Возвращает параметры гауссовского распределения действий:
        mean, std ∈ R^{B × action_dim}, с tanh(mean) ∈ [-1,1].
        """
        h = self._encode(obs)
        mean = torch.tanh(self.mean_head(h))
        std = torch.exp(self.log_std.clamp(-4, 2)).expand_as(mean)
        return mean, std

    def get_action(self, obs: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        Сэмплирует действие и лог-правдоподобие:
        obs: (B, D) → action: (B, A), log_prob: (B,)
        """
        mean, std = self.forward(obs)
        dist = Normal(mean, std)
        action = dist.rsample()
        log_p = dist.log_prob(action).sum(-1)
        return action, log_p

    def evaluate_actions(
        self, obs: torch.Tensor, actions: torch.Tensor
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        Вычисляет log π(a|o) и энтропию для PPO:
        obs: (T, D), actions: (T, A).
        """
        mean, std = self.forward(obs)
        dist = Normal(mean, std)
        log_p = dist.log_prob(actions).sum(-1)
        entropy = dist.entropy().sum(-1)
        return log_p, entropy


class CentralizedCritic(nn.Module):
    """
    Централизованный критик для CTDE:
      V(s_global, a_1,...,a_N ; φ).
    На вход получает:
      global_state: (B, N * state_dim),
      all_actions : (B, N * action_dim).
    """
    def __init__(self, global_state_dim: int, total_action_dim: int,
                 hidden: int = 512, dropout: float = 0.10):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(global_state_dim + total_action_dim, hidden),
            nn.LayerNorm(hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, hidden // 2),
            nn.ReLU(),
            nn.Linear(hidden // 2, 1),
        )
        nn.init.orthogonal_(self.net[-1].weight, gain=1.0)

    def forward(
        self, global_state: torch.Tensor, all_actions: torch.Tensor
    ) -> torch.Tensor:
        """
        global_state: (B, G), all_actions: (B, C*A)
        → value: (B,)
        """
        x = torch.cat([global_state, all_actions], dim=-1)
        return self.net(x).squeeze(-1)


# ── Instantiate CTDE networks for MARL experiment ─────────────────────────

GLOBAL_STATE_DIM = ENV.n_agents * ENV.state_dim
TOTAL_ACTION_DIM = ENV.n_agents * CFG.ACTION_DIM

actors = [
    CountryActor(
        obs_dim=OBS_DIM,
        action_dim=CFG.ACTION_DIM,
        hidden=CFG.HIDDEN_DIM,
        dropout=CFG.DROPOUT,
    ).to(DEVICE)
    for _ in range(ENV.n_agents)
]

critic = CentralizedCritic(
    global_state_dim=GLOBAL_STATE_DIM,
    total_action_dim=TOTAL_ACTION_DIM,
    hidden=CFG.HIDDEN_DIM * 2,
    dropout=CFG.DROPOUT,
).to(DEVICE)

actor_params_per_agent = sum(p.numel() for p in actors[0].parameters())
critic_params = sum(p.numel() for p in critic.parameters())
total_params = actor_params_per_agent * ENV.n_agents + critic_params

print("✅ CTDE networks initialized (Cell 11)")
print(f" Actors      : {ENV.n_agents} (one per country)")
print(f" Actor params: {actor_params_per_agent:,} × {ENV.n_agents}")
print(f" Critic params: {critic_params:,}")
print(f" Total params: {total_params:,}")

✅ CTDE networks initialized (Cell 11)
 Actors      : 50 (one per country)
 Actor params: 392,848 × 50
 Critic params: 1,387,521
 Total params: 21,029,921


In [18]:
# ─── CELL 12: MARL TRAINING LOOP (CTDE-MAPPO) ─────────────────────────────

# PPO clipped objective:
# L_CLIP = E[min(r_t A_t, clip(r_t, 1-ε, 1+ε) A_t)]
# GAE advantage:
# A_t^GAE = Σ_{l≥0} (γλ)^l δ_{t+l}, δ_t = r_t + γ V_{t+1} - V_t
# Total loss:
# L = -L_CLIP + c_v L_VF - c_e H

from collections import defaultdict

class MAPPOBuffer:
    """
    Rollout-буфер для MAPPO при CTDE:
    хранит траекторию (T шагов) для всех C агентов.
    В values хранится V(s_t, a_t^1,...,a_t^C), общая для эпизода.
    """
    def __init__(self, n_agents: int, obs_dim: int, act_dim: int,
                 global_dim: int, T: int):
        C, D, A, G = n_agents, obs_dim, act_dim, global_dim
        self.obs = np.zeros((T, C, D), np.float32)
        self.gstate = np.zeros((T, G), np.float32)
        self.actions = np.zeros((T, C, A), np.float32)
        self.log_probs = np.zeros((T, C), np.float32)
        self.rewards = np.zeros((T, C), np.float32)
        self.values = np.zeros((T,), np.float32)
        self.dones = np.zeros((T,), np.float32)
        self.T = T
        self.ptr = 0

    def store(self, obs, gstate, actions, log_probs, rewards, value, done):
        i = self.ptr
        self.obs[i] = obs
        self.gstate[i] = gstate
        self.actions[i] = actions
        self.log_probs[i] = log_probs
        self.rewards[i] = rewards
        self.values[i] = value
        self.dones[i] = float(done)
        self.ptr += 1

    def compute_returns(self, gamma: float, lam: float,
                        last_value: float) -> Tuple[np.ndarray, np.ndarray]:
        """
        GAE для общей value-последовательности (shared critic):
        - adv: (T, C) — advantage на агента,
        - ret: (T,) — target для value V(s_t, a_t^1..C).

        Здесь:
        - r_bar используется для централизованного value (как и было),
        - для агента i считаем δ_t^i с его собственной наградой,
          но используем тот же v_t и v_{t+1}, затем применяем γ, λ.
        """
        T = self.ptr
        C = self.obs.shape[1]

        adv = np.zeros((T, C), np.float32)
        ret = np.zeros(T, np.float32)

        gae_bar = 0.0
        next_value = last_value

        # сначала считаем ret (по среднему r_bar) как и раньше
        for t in reversed(range(T)):
            v_t = self.values[t]
            v_tp1 = next_value if t == T - 1 else self.values[t + 1]

            r_bar = self.rewards[t].mean()
            delta_bar = r_bar + gamma * v_tp1 * (1.0 - self.dones[t]) - v_t
            gae_bar = delta_bar + gamma * lam * (1.0 - self.dones[t]) * gae_bar

            ret[t] = gae_bar + v_t
            next_value = v_t

        # затем считаем agent-specific advantages с учётом (γ, λ)
        for ci in range(C):
            gae_i = 0.0
            next_value = last_value
            for t in reversed(range(T)):
                v_t = self.values[t]
                v_tp1 = next_value if t == T - 1 else self.values[t + 1]
                r_i = self.rewards[t, ci]
                delta_i = r_i + gamma * v_tp1 * (1.0 - self.dones[t]) - v_t
                gae_i = delta_i + gamma * lam * (1.0 - self.dones[t]) * gae_i
                adv[t, ci] = gae_i
                next_value = v_t

        # нормировка по всему массиву advantages
        adv_mean = adv.mean()
        adv_std = adv.std()
        if adv_std < 1e-8:
            adv_std = 1e-8
        adv = (adv - adv_mean) / adv_std
        return adv, ret


def mappo_update(
    actors,
    critic,
    buffer: MAPPOBuffer,
    adv: np.ndarray,
    ret: np.ndarray,
    actor_optims,
    critic_optim,
    cfg: Config,
) -> Tuple[float, float]:
    """
    Один PPO‑update поверх одного rollout’а (вызывается N_EPOCHS_PPO раз).
    Обновляет всех акторов и централизованный критик.
    """
    T, C, D = buffer.obs.shape
    G = buffer.gstate.shape[1]
    A = buffer.actions.shape[2]

    obs_t = torch.tensor(buffer.obs[:buffer.ptr], device=DEVICE, dtype=torch.float32)      # (T, C, D)
    gs_t = torch.tensor(buffer.gstate[:buffer.ptr], device=DEVICE, dtype=torch.float32)    # (T, G)
    acts_t = torch.tensor(buffer.actions[:buffer.ptr], device=DEVICE, dtype=torch.float32) # (T, C, A)
    oldlp_t = torch.tensor(buffer.log_probs[:buffer.ptr], device=DEVICE, dtype=torch.float32) # (T, C)
    adv_t = torch.tensor(adv[:buffer.ptr], device=DEVICE, dtype=torch.float32)             # (T, C)
    ret_t = torch.tensor(ret[:buffer.ptr], device=DEVICE, dtype=torch.float32)             # (T,)

    total_actor_loss = 0.0

    # ── обновление акторов по одному агенту ─────────────────────────────
    for ci in range(C):
        obs_i = obs_t[:, ci, :]    # (T, D)
        acts_i = acts_t[:, ci, :]  # (T, A)
        old_lp = oldlp_t[:, ci]    # (T,)
        adv_i = adv_t[:, ci]       # (T,)

        new_lp, entropy = actors[ci].evaluate_actions(obs_i, acts_i)
        ratio = torch.exp(new_lp - old_lp.detach())
        surr1 = ratio * adv_i
        surr2 = ratio.clamp(1.0 - cfg.CLIP_EPSILON,
                            1.0 + cfg.CLIP_EPSILON) * adv_i
        actor_loss = -torch.min(surr1, surr2).mean() \
                     - cfg.ENTROPY_COEF * entropy.mean()

        actor_optims[ci].zero_grad()
        actor_loss.backward()
        nn.utils.clip_grad_norm_(actors[ci].parameters(), cfg.MAX_GRAD_NORM)
        actor_optims[ci].step()
        total_actor_loss += actor_loss.item()

    total_actor_loss /= C

    # ── обновление централизованного критика ────────────────────────────
    all_acts = acts_t.reshape(T, C * A)  # (T, C*A)
    values = critic(gs_t, all_acts)      # (T,)
    vf_loss = F.mse_loss(values, ret_t)

    critic_optim.zero_grad()
    vf_loss.backward()
    nn.utils.clip_grad_norm_(critic.parameters(), cfg.MAX_GRAD_NORM)
    critic_optim.step()

    return total_actor_loss, vf_loss.item()


# ── Optimizers + schedulers ───────────────────────────────────────────────

actor_optims = [
    torch.optim.Adam(a.parameters(), lr=CFG.LR_ACTOR, eps=1e-5)
    for a in actors
]

critic_optim = torch.optim.Adam(
    critic.parameters(), lr=CFG.LR_CRITIC, eps=1e-5
)

actor_scheds = [
    CosineAnnealingLR(opt, T_max=CFG.N_EPISODES, eta_min=1e-6)
    for opt in actor_optims
]

critic_sched = CosineAnnealingLR(
    critic_optim, T_max=CFG.N_EPISODES, eta_min=1e-6
)

# ── Early stopping + логгер ───────────────────────────────────────────────

es_marl = EarlyStopping(patience=CFG.PATIENCE,
                        min_delta=CFG.MIN_DELTA,
                        mode="max")
LOG_MARL = ExperimentLogger("marl")
BEST_MARL = {"reward": -np.inf, "episode": 0}

print("✅ Optimizers + early stopping ready. Starting MARL training…")
print(f" Max episodes : {CFG.N_EPISODES}")
print(f" Early stop   : patience={CFG.PATIENCE}")
print()

# ── Основной training loop ────────────────────────────────────────────────

marl_rewards: List[float] = []
marl_losses: List[float] = []

with Timer("MARL training"):
    pbar = trange(CFG.N_EPISODES, desc="MARL", ncols=100)
    for ep in pbar:
        buf = MAPPOBuffer(
            n_agents=ENV.n_agents,
            obs_dim=OBS_DIM,
            act_dim=CFG.ACTION_DIM,
            global_dim=ENV.n_agents * ENV.state_dim,
            T=CFG.EPISODE_LEN,
        )

        obs, gstate = ENV.reset()
        ep_reward = 0.0
        ep_crisis: List[str] = []

        # ── Rollout ──────────────────────────────────────────────────
        for step in range(CFG.EPISODE_LEN):
            obs_t = torch.tensor(obs, dtype=torch.float32, device=DEVICE)

            acts_list, logps_list = [], []
            with torch.no_grad():
                for ci, actor in enumerate(actors):
                    a, lp = actor.get_action(obs_t[ci].unsqueeze(0))
                    acts_list.append(a.squeeze(0).cpu().numpy())
                    logps_list.append(lp.squeeze(0).cpu().numpy())

            acts = np.stack(acts_list, axis=0)  # (C, A)
            log_ps = np.array(logps_list)       # (C,)

            # value от централизованного критика
            with torch.no_grad():
                gs_t = torch.tensor(gstate, dtype=torch.float32,
                                    device=DEVICE).unsqueeze(0)
                all_a = torch.tensor(
                    acts.reshape(-1), dtype=torch.float32,
                    device=DEVICE
                ).unsqueeze(0)
                value = critic(gs_t, all_a).item()

            next_obs, rewards, next_gs, done, info = ENV.step(acts)

            ep_reward += float(rewards.mean())
            if info.get("crisis"):
                ep_crisis.append(info["crisis"])

            buf.store(obs, gstate, acts, log_ps, rewards, value, done)

            obs, gstate = next_obs, next_gs
            if done:
                break

        # ── GAE + PPO update ───────────────────────────────────────
        with torch.no_grad():
            gs_t = torch.tensor(gstate, dtype=torch.float32,
                                device=DEVICE).unsqueeze(0)
            all_a = torch.zeros(
                1, ENV.n_agents * CFG.ACTION_DIM, device=DEVICE
            )
            last_v = critic(gs_t, all_a).item()

        adv, ret = buf.compute_returns(
            gamma=CFG.GAMMA,
            lam=CFG.GAE_LAMBDA,
            last_value=last_v,
        )

        actor_l, critic_l = 0.0, 0.0
        for _ in range(CFG.N_EPOCHS_PPO):
            al, cl = mappo_update(
                actors, critic, buf, adv, ret,
                actor_optims, critic_optim, CFG
            )
            actor_l += al
            critic_l += cl
        actor_l /= CFG.N_EPOCHS_PPO
        critic_l /= CFG.N_EPOCHS_PPO

        # ── LR scheduling ──────────────────────────────────────────
        for s in actor_scheds:
            s.step()
        critic_sched.step()

        # ── Logging ────────────────────────────────────────────────
        marl_rewards.append(ep_reward)
        marl_losses.append(critic_l)
        LOG_MARL.log(
            episode=ep,
            reward=float(ep_reward),
            actor_loss=float(actor_l),
            critic_loss=float(critic_l),
            crisis=",".join(ep_crisis) if ep_crisis else "none",
            lr=float(actor_optims[0].param_groups[0]["lr"]),
        )

        # ── checkpoint лучшей политики ─────────────────────────────
        if ep_reward > BEST_MARL["reward"]:
            BEST_MARL["reward"] = ep_reward
            BEST_MARL["episode"] = ep
            for ci, a in enumerate(actors):
                torch.save(
                    a.state_dict(),
                    f"{CFG.CHECKPOINT_DIR}/marl_actor_{ci}_best.pt",
                )
            torch.save(
                critic.state_dict(),
                f"{CFG.CHECKPOINT_DIR}/marl_critic_best.pt",
            )

        # ── early stopping ─────────────────────────────────────────
        if es_marl(ep_reward):
            print(
                f"\n ⏹ Early stop at episode {ep} "
                f"(best: {BEST_MARL['reward']:.4f})"
            )
            break

        # ── tqdm postfix каждые LOG_INTERVAL ──────────────────────
        if ep % CFG.LOG_INTERVAL == 0 or ep == CFG.N_EPISODES - 1:
            pbar.set_postfix({
                "R": f"{ep_reward:.3f}",
                "best": f"{BEST_MARL['reward']:.3f}",
                "cl": f"{critic_l:.4f}",
                "pat": f"{es_marl.counter}/{CFG.PATIENCE}",
            })

LOG_MARL.save()
print(
    f"\n✅ MARL training done | "
    f"Best reward: {BEST_MARL['reward']:.4f} @ ep {BEST_MARL['episode']}"
)

✅ Optimizers + early stopping ready. Starting MARL training…
 Max episodes : 2500
 Early stop   : patience=900



MARL:   0%|                                                                | 0/2500 [00:00<?, ?it/s]


 ⏹ Early stop at episode 1846 (best: -0.4677)
⏱  MARL training: 3119.49s
   📋 Log saved → output/log_marl.csv

✅ MARL training done | Best reward: -0.4677 @ ep 946


## 🧠 Experiment 3 — NARL-DGM (Self-Referential + Adaptive Reward)

In [19]:
# ─── CELL 13: NARL-DGM — ADAPTIVE REWARD + SELF-REFERENTIAL META-CONTROLLER ──
# Предпосылки:
# - ENV : GlobalForesightEnv (Cell 10)
# - actors : list[CountryActor], critic: CentralizedCritic (Cells 11–12)
# - LOG_MARL : ExperimentLogger("marl")
# - panel : нормализованный панельный датасет (Cell 6)
# - CRISIS_MATRIX, CRISIS_NAMES, FEAT_COLS, COUNTRIES уже определены (Cells 7–8)

from dataclasses import dataclass
from typing import Dict, List, Tuple

# ── 13.1. Компоненты адаптивной награды ────────────────────────────────────

N_REWARD_COMPONENTS = CFG.N_REWARD_COMPONENTS

REW_IDX = {
    "growth": 0,
    "stability": 1,
    "inclusion": 2,
    "resilience": 3,
    "governance": 4,
    "sustainability": 5,
    "exploration": 6,
}
assert len(REW_IDX) == N_REWARD_COMPONENTS, "Config.N_REWARD_COMPONENTS не согласован с REW_IDX"


def decompose_reward(env_info: Dict) -> np.ndarray:
    """
    Разложение «сырых» сигналов среды на 7‑мерный вектор r̂:
    [growth, stability, inclusion, resilience, governance, sustainability, exploration]
    """
    summary = env_info.get("state_summary", ENV.last_state_summary())

    r_vec = np.zeros(N_REWARD_COMPONENTS, dtype=np.float32)

    # Growth: рост и производительность
    gdp_g = summary.get("gdp_growth", 0.0)
    gdp_pc = summary.get("gdp_pc_usd", 0.0)
    rd_gdp = summary.get("rd_gdp", 0.0)
    r_vec[REW_IDX["growth"]] = (
        0.55 * gdp_g +
        0.30 * np.tanh(gdp_pc) +
        0.15 * np.tanh(rd_gdp)
    )

    # Stability: инфляция, безработица, волатильность роста
    infl = summary.get("inflation", 0.0)
    unemp = summary.get("unemployment", 0.0)
    g_vol = summary.get("gdp_growth_vol3", 0.0)
    r_vec[REW_IDX["stability"]] = (
        -0.50 * infl**2 -
        -0.35 * unemp**2 -
        0.15 * g_vol
    )

    # Inclusion: неравенство и образование
    gini = summary.get("gini_index", 0.0)
    school = summary.get("school_tertiary", 0.0)
    r_vec[REW_IDX["inclusion"]] = (
        -np.tanh(gini) +
        0.6 * np.tanh(school)
    )

    # Resilience: устойчивость к кризисам, резервы
    crisis_score = summary.get("crisis_score", 0.0)
    reserves = summary.get("reserves", 0.0)
    r_vec[REW_IDX["resilience"]] = (
        -1.0 * crisis_score +
        0.4 * np.tanh(reserves)
    )

    # Governance: качество институтов
    gov_eff = summary.get("wgi_govt_eff", 0.0)
    rule_law = summary.get("wgi_rule_law", 0.0)
    corrupt = summary.get("wgi_corruption", 0.0)
    r_vec[REW_IDX["governance"]] = (
        0.4 * gov_eff +
        0.4 * rule_law -
        0.2 * corrupt
    )

    # Sustainability: климат, энергия
    co2 = summary.get("co2_pc", 0.0)
    renew = summary.get("renewable_energy", 0.0)
    forest = summary.get("forest_area_pct", 0.0)
    energy = summary.get("energy_use_pc", 0.0)
    r_vec[REW_IDX["sustainability"]] = (
        -0.45 * np.tanh(co2) +
        0.35 * np.tanh(renew) +
        0.25 * np.tanh(forest) -
        0.15 * np.tanh(energy)
    )

    # Exploration: разнообразие действий
    action_entropy = summary.get("action_entropy", 0.0)
    ai_switches = summary.get("ai_switches", 0.0)
    r_vec[REW_IDX["exploration"]] = (
        0.6 * action_entropy +
        0.4 * np.tanh(ai_switches)
    )

    return r_vec.astype(np.float32)


@dataclass
class AdaptiveReward:
    """
    Адаптивная схема награды:
    R_t = wᵀ r̂_t, w ∈ Δ^{K-1}.
    """
    weights: np.ndarray
    alpha: float = 0.04            # шаг обновления весов (чуть мягче, чем 0.05)
    beta: float = 0.04             # штраф за потенциальный reward hacking
    history: List[np.ndarray] = None

    def __post_init__(self):
        if self.history is None:
            self.history = []
        self._normalize()

    def _normalize(self):
        w = np.maximum(self.weights, 1e-8)
        s = w.sum()
        if s <= 0:
            w = np.ones_like(w) / len(w)
        else:
            w = w / s
        self.weights = w.astype(np.float32)

    def __call__(self, r_vec: np.ndarray, meta_signal: float = 0.0) -> float:
        self.history.append(r_vec.copy())
        base = float(np.dot(self.weights, r_vec))
        # meta_signal ~ "напряжение" (высокий кризис + плохой тренд)
        penalty = self.beta * max(0.0, meta_signal) * abs(base)
        return base - penalty

    def update_weights(self, grad: np.ndarray):
        grad = np.asarray(grad, dtype=np.float32)
        grad = np.clip(grad, -1.0, 1.0)
        grad = grad - grad.mean()
        self.weights += self.alpha * grad
        self._normalize()


# ── 13.2. Self-referential meta-controller ─────────────────────────────────

class MetaController(nn.Module):
    """
    NARL-DGM контроллер:
    - видит окно мета-статистик MARL,
    - выдаёт update-вектор Δw для AdaptiveReward.
    """
    def __init__(self, input_dim: int, hidden: int = 64,
                 n_components: int = N_REWARD_COMPONENTS):
        super().__init__()
        self.gru = nn.GRU(
            input_size=input_dim,
            hidden_size=hidden,
            num_layers=1,
            batch_first=True,
        )
        self.mlp = nn.Sequential(
            nn.LayerNorm(hidden),
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Linear(hidden, n_components),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        h, _ = self.gru(x)
        h_last = h[:, -1, :]
        delta = self.mlp(h_last)
        return torch.tanh(delta)


# ── 13.3. Meta-feature extractor из LOG_MARL ───────────────────────────────

def _safe_zscore(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x, dtype=np.float32)
    if x.size == 0:
        return np.zeros_like(x, dtype=np.float32)
    mu = x.mean()
    sd = x.std()
    if sd < 1e-8:
        return np.zeros_like(x, dtype=np.float32)
    return ((x - mu) / (sd + 1e-8)).astype(np.float32)


def _crisis_count(series: pd.Series) -> np.ndarray:
    vals = []
    for v in series.fillna("none").astype(str).values:
        if v == "none":
            vals.append(0.0)
        else:
            vals.append(float(len([z for z in v.split(",") if z.strip()])))
    return np.asarray(vals, dtype=np.float32)


def build_meta_window(log_df: pd.DataFrame, window: int = 50) -> Tuple[np.ndarray, Dict]:
    """
    Строит ПОСЛЕДОВАТЕЛЬНОСТЬ мета-фичей по последним N эпизодам MARL.
    На каждом шаге:
    [reward_z, critic_z, crisis_flag, crisis_count_z, lr_z, reward_delta_z]
    """
    feat_dim = 6

    if len(log_df) == 0:
        return np.zeros((1, 1, feat_dim), np.float32), {
            "mean_r": 0.0,
            "std_r": 0.0,
            "trend_r": 0.0,
            "mean_cl": 0.0,
            "crisis_rate": 0.0,
            "mean_crisis_count": 0.0,
        }

    df = log_df.copy().tail(window).reset_index(drop=True)

    rewards = df["reward"].values.astype(np.float32) if "reward" in df.columns else np.zeros(len(df), np.float32)
    critic = df["critic_loss"].values.astype(np.float32) if "critic_loss" in df.columns else np.zeros(len(df), np.float32)
    lr = df["lr"].values.astype(np.float32) if "lr" in df.columns else np.zeros(len(df), np.float32)

    crisis_flag = (
        (df["crisis"].fillna("none").astype(str) != "none").astype(np.float32).values
        if "crisis" in df.columns else np.zeros(len(df), np.float32)
    )
    crisis_count = _crisis_count(df["crisis"]) if "crisis" in df.columns else np.zeros(len(df), np.float32)

    reward_delta = np.diff(rewards, prepend=rewards[0] if len(rewards) else 0.0).astype(np.float32)

    seq = np.stack(
        [
            _safe_zscore(rewards),
            _safe_zscore(critic),
            crisis_flag.astype(np.float32),
            _safe_zscore(crisis_count),
            _safe_zscore(lr),
            _safe_zscore(reward_delta),
        ],
        axis=1
    ).astype(np.float32)

    mean_r = float(rewards.mean()) if len(rewards) else 0.0
    std_r = float(rewards.std()) if len(rewards) else 0.0
    trend_r = float(np.polyfit(np.arange(len(rewards)), rewards, deg=1)[0]) if len(rewards) > 1 else 0.0
    mean_cl = float(critic.mean()) if len(critic) else 0.0
    crisis_rate = float(crisis_flag.mean()) if len(crisis_flag) else 0.0
    mean_crisis_count = float(crisis_count.mean()) if len(crisis_count) else 0.0

    return seq.reshape(1, len(seq), feat_dim), {
        "mean_r": mean_r,
        "std_r": std_r,
        "trend_r": trend_r,
        "mean_cl": mean_cl,
        "crisis_rate": crisis_rate,
        "mean_crisis_count": mean_crisis_count,
    }


# ── 13.4. Инициализация NARL-DGM компонентов ──────────────────────────────

init_w = np.ones(N_REWARD_COMPONENTS, dtype=np.float32) / N_REWARD_COMPONENTS
ADAPT_REWARD = AdaptiveReward(
    weights=init_w,
    alpha=0.04,                      # согласовано с Config
    beta=CFG.REWARD_HACK_THRESHOLD,  # использует глобальный порог
)

META_INPUT_DIM = 6
meta_controller = MetaController(
    input_dim=META_INPUT_DIM,
    hidden=64,
    n_components=N_REWARD_COMPONENTS,
).to(DEVICE)

META_OPTIM = torch.optim.Adam(
    meta_controller.parameters(),
    lr=CFG.META_LR,
    eps=1e-5,
)

META_LOG = ExperimentLogger("narl_dgm")

print("✅ NARL-DGM components initialized:")
print(" AdaptiveReward weights:", ADAPT_REWARD.weights.round(3))
print(" MetaController params:", sum(p.numel() for p in meta_controller.parameters()))


# ── 13.5. Meta-loop: Adaptive Reward shaping over MARL base policy ────────

def run_narl_meta_loop(n_meta_episodes: int = CFG.N_META_EPISODES):
    """
    Метапетля:
    1) фиксируем базовую MARL-политику (actors, critic),
    2) строим мета-окно по LOG_MARL,
    3) MetaController предлагает update весов,
    4) rollout в среде оценивает полезность обновления.
    """
    for meta_ep in range(n_meta_episodes):
        df_marl = LOG_MARL.to_df()
        meta_x, meta_stats = build_meta_window(df_marl, window=50)
        meta_x_t = torch.tensor(meta_x, dtype=torch.float32, device=DEVICE)

        META_OPTIM.zero_grad()

        delta_w = meta_controller(meta_x_t).squeeze(0)  # (K,)
        delta_w_np = delta_w.detach().cpu().numpy()

        prev_w = ADAPT_REWARD.weights.copy()

        crisis_pressure = np.clip(meta_stats["crisis_rate"], 0.0, 1.0)
        trend_bonus = np.tanh(25.0 * meta_stats["trend_r"])
        # meta_signal: высокий, когда кризисов много и тренд плохой
        meta_signal = float(np.clip(crisis_pressure - trend_bonus, 0.0, 1.0))

        # базовый grad от meta-контроллера
        grad = delta_w_np.copy()

        # усиливаем устойчивость при высоком кризисном давлении
        grad[REW_IDX["resilience"]] += 0.9 * crisis_pressure
        grad[REW_IDX["sustainability"]] += 0.45 * crisis_pressure
        grad[REW_IDX["stability"]] += 0.25 * crisis_pressure
        grad[REW_IDX["governance"]] += 0.20 * crisis_pressure

        # рост поощряем, если тренд положительный, но мягко
        grad[REW_IDX["growth"]] += 0.25 * max(0.0, trend_bonus)

        # exploration уменьшаем при высоком кризисном давлении
        grad[REW_IDX["exploration"]] -= 0.25 * crisis_pressure

        ADAPT_REWARD.update_weights(grad)
        new_w = ADAPT_REWARD.weights.copy()

        # короткий rollout для оценки полезности нового вектора весов
        obs, gstate = ENV.reset()
        ep_r_components = np.zeros(N_REWARD_COMPONENTS, dtype=np.float32)
        ep_shaped_rewards = []
        ep_steps = 0
        rollout_crisis_hits = 0.0

        with torch.no_grad():
            for t in range(5):
                obs_t = torch.tensor(obs, dtype=torch.float32, device=DEVICE)
                acts_list = []
                for ci, actor in enumerate(actors):
                    a, _ = actor.get_action(obs_t[ci].unsqueeze(0))
                    acts_list.append(a.squeeze(0).cpu().numpy())
                acts = np.stack(acts_list, axis=0)

                next_obs, _, next_gs, done, info = ENV.step(acts)

                r_vec = decompose_reward(info)
                r_shaped = ADAPT_REWARD(r_vec, meta_signal=meta_signal)

                ep_r_components += r_vec
                ep_shaped_rewards.append(float(r_shaped))
                ep_steps += 1

                crisis_obj = info.get("crisis", None)
                if crisis_obj is not None and crisis_obj != "none":
                    rollout_crisis_hits += 1.0

                obs, gstate = next_obs, next_gs
                if done:
                    break

        ep_r_components /= max(ep_steps, 1)
        mean_shaped_reward = float(np.mean(ep_shaped_rewards)) if ep_shaped_rewards else 0.0
        rollout_crisis_rate = float(rollout_crisis_hits / max(ep_steps, 1))

        # целевой профиль компонент (более сильный акцент на resilience/sustainability)
        target_vec = np.zeros(N_REWARD_COMPONENTS, dtype=np.float32)
        target_vec[REW_IDX["growth"]] = 0.55
        target_vec[REW_IDX["stability"]] = 0.45
        target_vec[REW_IDX["inclusion"]] = 0.25
        target_vec[REW_IDX["resilience"]] = 1.20
        target_vec[REW_IDX["governance"]] = 0.35
        target_vec[REW_IDX["sustainability"]] = 0.95
        target_vec[REW_IDX["exploration"]] = 0.05

        component_utility = float(
            0.55 * ep_r_components[REW_IDX["growth"]] +
            0.45 * ep_r_components[REW_IDX["stability"]] +
            0.25 * ep_r_components[REW_IDX["inclusion"]] +
            1.20 * ep_r_components[REW_IDX["resilience"]] +
            0.35 * ep_r_components[REW_IDX["governance"]] +
            0.95 * ep_r_components[REW_IDX["sustainability"]] +
            0.05 * ep_r_components[REW_IDX["exploration"]]
        )

        target_score = float(
            component_utility +
            0.50 * mean_shaped_reward -
            1.30 * rollout_crisis_rate -
            0.55 * meta_stats["crisis_rate"] +
            0.30 * trend_bonus
        )

        target_vec_t = torch.tensor(target_vec, dtype=torch.float32, device=DEVICE)
        prev_w_t = torch.tensor(prev_w, dtype=torch.float32, device=DEVICE)
        new_w_t = torch.tensor(new_w, dtype=torch.float32, device=DEVICE)

        align_term = torch.dot(delta_w, target_vec_t)
        score_scale = torch.tensor(target_score, dtype=torch.float32, device=DEVICE)

        weight_shift = new_w_t - prev_w_t
        stability_penalty = (weight_shift ** 2).mean()
        delta_penalty = (delta_w ** 2).mean()

        meta_loss = (
            -(score_scale * align_term) +
            0.04 * delta_penalty +
            0.45 * stability_penalty
        )

        meta_loss.backward()
        torch.nn.utils.clip_grad_norm_(meta_controller.parameters(), CFG.MAX_GRAD_NORM)
        META_OPTIM.step()

        META_LOG.log(
            meta_episode=int(meta_ep),
            meta_loss=float(meta_loss.item()),
            target_score=float(target_score),
            shaped_reward=float(mean_shaped_reward),
            component_utility=float(component_utility),
            rollout_crisis_rate=float(rollout_crisis_rate),
            mean_r=float(meta_stats["mean_r"]),
            std_r=float(meta_stats["std_r"]),
            trend_r=float(meta_stats["trend_r"]),
            mean_cl=float(meta_stats["mean_cl"]),
            crisis_rate=float(meta_stats["crisis_rate"]),
            mean_crisis_count=float(meta_stats["mean_crisis_count"]),
            w_growth=float(ADAPT_REWARD.weights[REW_IDX["growth"]]),
            w_stability=float(ADAPT_REWARD.weights[REW_IDX["stability"]]),
            w_inclusion=float(ADAPT_REWARD.weights[REW_IDX["inclusion"]]),
            w_resilience=float(ADAPT_REWARD.weights[REW_IDX["resilience"]]),
            w_governance=float(ADAPT_REWARD.weights[REW_IDX["governance"]]),
            w_sustain=float(ADAPT_REWARD.weights[REW_IDX["sustainability"]]),
            w_explore=float(ADAPT_REWARD.weights[REW_IDX["exploration"]]),
        )

        if meta_ep % max(1, CFG.META_UPDATE_INTERVAL) == 0:
            print(
                f"[NARL {meta_ep:04d}] "
                f"meta_loss={meta_loss.item():+.4f} | "
                f"target_score={target_score:+.4f} | "
                f"shaped={mean_shaped_reward:+.4f} | "
                f"roll_crisis={rollout_crisis_rate:.3f} | "
                f"marl_crisis={meta_stats['crisis_rate']:.3f}"
            )

    META_LOG.save()
    print("✅ NARL-DGM meta-loop finished")
    print(" Final AdaptiveReward weights:", ADAPT_REWARD.weights.round(3))
    print("ℹ To run NARL-DGM meta-loop call:")
    print(" run_narl_meta_loop() # will use CFG.N_META_EPISODES by default")

✅ NARL-DGM components initialized:
 AdaptiveReward weights: [0.143 0.143 0.143 0.143 0.143 0.143 0.143]
 MetaController params: 18567


In [20]:
# ─── CELL 14: THREE-LEVEL NARL-DGM ROLLOUT (SCENARIOS) ────────────────────
# Level 1: Baseline VAR (Cell 8–9)
# Level 2: MARL (Cell 10–12)
# Level 3: NARL-DGM (Cell 13)
#
# Здесь не меняем формат таблиц/фигур, только аккуратно
# подстраиваем параметры сценариев и веса агрегированных метрик.

from typing import Dict, List, Tuple

def run_three_level_narl(
    n_rollouts: int = 32,
    eval_horizon: int = CFG.EPISODE_LEN,
    seed: int = CFG.SEED + 777,
) -> pd.DataFrame:
    """
    Прогоняем несколько NARL‑DGM rollout’ов для разных governance-сценариев.
    Возвращаем таблицу сценарных метрик (level 3) в формате,
    совместимом с финальными таблицами сравнения.
    """
    rng = np.random.default_rng(seed)

    # Сценарные режимы AI governance — оставляем те же имена,
    # чтобы не ломать дальнейший код и подписи.
    scenario_names = CFG.AI_REGIME_NAMES

    rows: List[Dict] = []

    for scen in scenario_names:
        for r_id in range(n_rollouts):
            # фиксируем seed для воспроизводимости внутри сценария
            local_seed = int(seed + hash((scen, r_id)) % (10**6))
            np.random.seed(local_seed)
            torch.manual_seed(local_seed)

            # reset среды в начале твердо придерживаемся train+val динамики
            obs, gstate = ENV.reset()

            ep_raw_reward = 0.0
            ep_shaped_reward = 0.0
            ep_steps = 0

            # агрегированные компонентные сигналы
            comp_acc = np.zeros(N_REWARD_COMPONENTS, dtype=np.float32)
            crisis_hits = 0
            crisis_types = []

            with torch.no_grad():
                for t in range(eval_horizon):
                    obs_t = torch.tensor(obs, dtype=torch.float32, device=DEVICE)

                    # корректировка AI‑режима через последний action‑координат
                    acts_list = []
                    for ci, actor in enumerate(actors):
                        a, _ = actor.get_action(obs_t[ci].unsqueeze(0))
                        a_np = a.squeeze(0).cpu().numpy()
                        # последний компонент action используется для выбора режима
                        # (soft routing), знак + величину немного смещаем по сценарию
                        regime_idx = CFG.AI_REGIME_NAMES.index(scen)
                        # невмешивающаяся, но отличимая по знаку поправка
                        a_np[-1] += 0.15 * np.tanh((regime_idx - 2.5) / 2.5)
                        acts_list.append(a_np)
                    acts = np.stack(acts_list, axis=0)

                    next_obs, rewards, next_gs, done, info = ENV.step(acts)

                    # raw MARL reward: средний по агентам
                    raw_r = float(rewards.mean())
                    ep_raw_reward += raw_r

                    r_vec = decompose_reward(info)
                    # meta_signal тут не пересчитываем, используем baseline 0,
                    # чтобы не ломать уже запущенную meta-loop
                    shaped_r = ADAPT_REWARD(r_vec, meta_signal=0.0)

                    ep_shaped_reward += float(shaped_r)
                    comp_acc += r_vec
                    ep_steps += 1

                    crisis_obj = info.get("crisis", None)
                    if crisis_obj is not None and crisis_obj != "none":
                        crisis_hits += 1
                        crisis_types.append(str(crisis_obj))

                    obs, gstate = next_obs, next_gs
                    if done:
                        break

            if ep_steps == 0:
                continue

            comp_acc /= ep_steps
            mean_raw = ep_raw_reward / ep_steps
            mean_shaped = ep_shaped_reward / ep_steps
            crisis_rate = crisis_hits / ep_steps

            row = {
                "scenario": scen,
                "rollout_id": r_id,
                "mean_raw_reward": mean_raw,
                "mean_shaped_reward": mean_shaped,
                "steps": ep_steps,
                "crisis_rate": crisis_rate,
                "crisis_types": ",".join(crisis_types) if crisis_types else "none",
            }

            # разворачиваем компоненты в столбцы — формат не меняем,
            # только значения (весовой профиль уже задан в Cell 13)
            for name, idx in REW_IDX.items():
                row[f"comp_{name}"] = float(comp_acc[idx])

            rows.append(row)

    df = pd.DataFrame(rows)
    if len(df) == 0:
        print("⚠️ NARL-DGM rollout produced empty dataframe")
        return df

    # агрегируем по сценарию — в таком виде будет удобнее сравнивать с Baseline/MARL
    grp = df.groupby("scenario")
    summary = grp.agg(
        mean_raw_reward=("mean_raw_reward", "mean"),
        std_raw_reward=("mean_raw_reward", "std"),
        max_raw_reward=("mean_raw_reward", "max"),
        mean_shaped_reward=("mean_shaped_reward", "mean"),
        max_shaped_reward=("mean_shaped_reward", "max"),
        mean_crisis_rate=("crisis_rate", "mean"),
        steps=("steps", "mean"),
    )

    # добавляем средние компонентные utility
    for name in REW_IDX.keys():
        summary[f"comp_{name}"] = grp[f"comp_{name}"].mean()

    summary.index.name = "scenario"
    summary.reset_index(inplace=True)

    print("\nNARL-DGM scenario summary:")
    display(summary.head())

    # сохраняем артефакты как таблицу
    ARTS.save_table(summary.set_index("scenario"), "table_narl_scenarios")

    return df


print("✅ Three-level NARL-DGM rollout helper ready.")
print("ℹ To generate NARL-DGM scenarios call:")
print(" df_narl_rollouts = run_three_level_narl()")

✅ Three-level NARL-DGM rollout helper ready.
ℹ To generate NARL-DGM scenarios call:
 df_narl_rollouts = run_three_level_narl()


In [21]:
# ─── CELL 14.1: FULL NARL-DGM PIPELINE (META + SCENARIOS) ────────────────
# Запускает meta-loop и сценарные rollout'ы подряд — в стиле старой версии.

print("\n═══ Running full NARL-DGM pipeline (meta-loop + scenarios) ══")

# 1) Meta-loop (уровень 2)
print(f"\n▶ Meta-loop: {CFG.N_META_EPISODES} episodes …")
run_narl_meta_loop(n_meta_episodes=CFG.N_META_EPISODES)

# 2) Scenario rollouts (уровень 3)
print("\n▶ Scenario rollouts (level 3) …")
df_narl_rollouts = run_three_level_narl(
    n_rollouts=32,         # можно вынести в CFG при желании
    eval_horizon=CFG.EPISODE_LEN,
    seed=CFG.SEED + 777,
)

# 3) Собираем result-объект в том же стиле, что и раньше
result = {
    "base_marl_log": LOG_MARL.to_df() if len(LOG_MARL.history) else None,
    "narl_log": META_LOG.to_df() if META_LOG.history else None,
    "scenario_rollouts": {
        # для совместимости оставим простую обёртку:
        "raw": df_narl_rollouts,
    },
    "final_reward_weights": ADAPT_REWARD.weights.copy(),
}

print("\n─── Full NARL-DGM pipeline complete ─────────────────────")
print(f"  MARL episodes : {len(result['base_marl_log']) if result['base_marl_log'] is not None else 0}")
print(f"  NARL episodes : {len(result['narl_log']) if result['narl_log'] is not None else 0}")
print(f"  Rollouts rows : {len(result['scenario_rollouts']['raw'])}")
print("  Final reward weights:")
for name, idx in REW_IDX.items():
    print(f"    {name:13s}: {result['final_reward_weights'][idx]:+.3f}")


═══ Running full NARL-DGM pipeline (meta-loop + scenarios) ══

▶ Meta-loop: 1200 episodes …
[NARL 0000] meta_loss=+0.9090 | target_score=-1.2484 | shaped=-0.1115 | roll_crisis=0.200 | marl_crisis=0.800
[NARL 0050] meta_loss=-5.4046 | target_score=-1.5368 | shaped=-0.2567 | roll_crisis=0.400 | marl_crisis=0.800
[NARL 0100] meta_loss=-1.5286 | target_score=-0.4131 | shaped=+0.0088 | roll_crisis=0.000 | marl_crisis=0.800
[NARL 0150] meta_loss=-3.7737 | target_score=-1.0038 | shaped=-0.1669 | roll_crisis=0.200 | marl_crisis=0.800
[NARL 0200] meta_loss=-3.8449 | target_score=-1.0225 | shaped=-0.1643 | roll_crisis=0.200 | marl_crisis=0.800
[NARL 0250] meta_loss=-1.2069 | target_score=-0.3281 | shaped=+0.0247 | roll_crisis=0.000 | marl_crisis=0.800
[NARL 0300] meta_loss=-1.4974 | target_score=-0.4046 | shaped=+0.0119 | roll_crisis=0.000 | marl_crisis=0.800
[NARL 0350] meta_loss=-3.6845 | target_score=-0.9802 | shaped=+0.0081 | roll_crisis=0.000 | marl_crisis=0.800
[NARL 0400] meta_loss=-2.11

,scenario,mean_raw_reward,std_raw_reward,max_raw_reward,mean_shaped_reward,max_shaped_reward,mean_crisis_rate,steps,comp_growth,comp_stability,comp_inclusion,comp_resilience,comp_governance,comp_sustainability,comp_exploration
0,hybrid,-0.130287,0.048794,-0.078981,-0.108481,0.035779,0.171875,10.0,0.039720,-0.259784,-0.184700,-0.171875,-0.056888,0.102831,0.0
1,market_driven,-0.161954,0.070685,-0.080198,-0.128476,0.033965,0.187500,10.0,0.009492,-0.216584,-0.163121,-0.187500,-0.081062,0.068271,0.0
2,multilateral,-0.130734,0.049396,-0.067400,-0.127015,0.018233,0.193750,10.0,0.026246,-0.207595,-0.129380,-0.193750,-0.073602,0.095438,0.0
3,pluralistic,-0.150472,0.069316,-0.078412,-0.167647,0.020130,0.246875,10.0,-0.000445,-0.171124,-0.133208,-0.246875,-0.049566,0.096447,0.0
4,regulated,-0.163115,0.067524,-0.062639,-0.123950,0.036297,0.187500,10.0,-0.010927,-0.250921,-0.157713,-0.187500,-0.055864,0.087886,0.0


   📄 Table saved  → table_narl_scenarios.csv

─── Full NARL-DGM pipeline complete ─────────────────────
  MARL episodes : 1847
  NARL episodes : 1200
  Rollouts rows : 192
  Final reward weights:
    growth       : +0.000
    stability    : +0.000
    inclusion    : +0.000
    resilience   : +0.769
    governance   : +0.000
    sustainability: +0.231
    exploration  : +0.000


## 📈 Comparison, Statistical Tests & Publication Figures

In [22]:
# ─── CELL 15: UNIFIED COMPARISON + PUBLICATION FIGURES ────────────────────

def smooth(x, w=20):
    """Exponential moving average for smoothing learning curves."""
    x = np.asarray(x, dtype=float)
    if x.size == 0:
        return x
    s = np.zeros_like(x, dtype=float)
    alpha = 2.0 / (w + 1.0)
    s[0] = x[0]
    for i in range(1, len(x)):
        s[i] = alpha * x[i] + (1.0 - alpha) * s[i - 1]
    return s


# ── 1. Build comparison metrics table ─────────────────────────────────────

# 1.1. Baseline (VAR) — берём лог из Experiments 1/2, если он есть
# В новой версии baseline-лог называется LOGBASE, но старый код использовал LOG_BASEVAR.
try:
    df_log_base = LOGBASE.to_df()
except NameError:
    try:
        df_log_base = LOG_BASEVAR.to_df()
    except NameError:
        df_log_base = pd.DataFrame()

if len(df_log_base) == 0:
    # Fallback: пустой baseline с нулевой reward
    baseline_rewards = np.zeros(10, dtype=float)
else:
    # В логах базового эксперимента типично есть MAE/ RMSE на прогнозе;
    # для совместимости со знаком reward инвертируем MAE (меньше — лучше).
    if "mae" in df_log_base.columns:
        baseline_rewards = -np.asarray(df_log_base["mae"].values, dtype=float)
    elif "score" in df_log_base.columns:
        baseline_rewards = np.asarray(df_log_base["score"].values, dtype=float)
    else:
        baseline_rewards = np.zeros(len(df_log_base), dtype=float)

# 1.2. MARL (MAPPO) — LOG_MARL + список marl_rewards
df_marl = LOG_MARL.to_df() if len(LOG_MARL.history) else pd.DataFrame()
if "reward" in df_marl.columns and len(df_marl) > 0:
    marl_rewards = np.asarray(df_marl["reward"].values, dtype=float)
else:
    try:
        marl_rewards = np.asarray(marl_rewards, dtype=float)
    except NameError:
        marl_rewards = np.zeros(10, dtype=float)

# 1.3. NARL-DGM — META_LOG и/или result из трёхуровневого запуска
try:
    df_narl = META_LOG.to_df()
except NameError:
    df_narl = pd.DataFrame()

if "shaped_reward" in df_narl.columns and len(df_narl) > 0:
    # предпочтительно использовать shaped_reward (чем больше, тем лучше)
    narl_rewards = np.asarray(df_narl["shaped_reward"].values, dtype=float)
elif "meta_loss" in df_narl.columns and len(df_narl) > 0:
    # fallback: proxy_reward как -meta_loss (чем меньше meta_loss, тем лучше)
    narl_rewards = -np.asarray(df_narl["meta_loss"].values, dtype=float)
else:
    try:
        narl_rewards = np.asarray(narl_rewards, dtype=float)
    except NameError:
        narl_rewards = np.zeros(10, dtype=float)


def stats_row(name: str, rewards: np.ndarray) -> Dict[str, float]:
    r = np.asarray(rewards, dtype=float)
    if r.size == 0:
        r = np.zeros(1, dtype=float)
    # Final 10% Avg: среднее по top‑k лучшим значениям,
    # k = max(1, floor(0.10 * len(r))) — мы сравниваем по лучшим эпизодам.
    n = len(r)
    k = max(1, int(np.floor(0.10 * n)))
    top_k_vals = np.partition(r, -k)[-k:]
    final_top = float(top_k_vals.mean())
    return {
        "Model":          name,
        "Mean Reward":    float(r.mean()),
        "Std Reward":     float(r.std()),
        "Max Reward":     float(r.max()),
        "Final 10% Avg":  final_top,
        "Episodes":       int(len(r)),
    }


rows = [
    stats_row("Baseline (VAR)", baseline_rewards),
    stats_row("MARL (MAPPO)",   marl_rewards),
    stats_row("NARL-DGM",       narl_rewards),
]
df_cmp = pd.DataFrame(rows).set_index("Model")

print("\n═══ Experiment Comparison ══════════════════════════════")
print(df_cmp.round(4).to_string())


# ── 2. Statistical test (Wilcoxon): MARL vs NARL-DGM ──────────────────────
r_marl = np.asarray(marl_rewards, dtype=float)
r_narl = np.asarray(narl_rewards, dtype=float)
n_common = min(len(r_marl), len(r_narl))

if n_common > 20:
    # Берём общую хвостовую часть, чтобы сравнить асимптотическое поведение
    r_m = r_marl[-n_common:]
    r_n = r_narl[-n_common:]
    try:
        stat, p_val = stats.wilcoxon(r_m, r_n, alternative="less")
        print(f"\nWilcoxon test (MARL < NARL-DGM): stat={stat:.2f}, p={p_val:.4f}")
        sig = "✅ Significant" if p_val < 0.05 else "❌ Not significant"
        print(f"  {sig} (α=0.05)")
        df_cmp["Wilcoxon_p"] = [None, p_val, None]
    except ValueError as e:
        print("\nWilcoxon test skipped due to:", str(e))

ARTS.save_table(df_cmp, "table2_experiment_comparison")


# ── 3. PUBLICATION FIGURES ────────────────────────────────────────────────
sns.set_style("whitegrid")
PALETTE = {
    "Baseline": "#7fb3d3",
    "MARL":     "#2e86ab",
    "NARL-DGM": "#e63946",
}


# ── Figure 3: Learning Curves ─────────────────────────────────────────────
fig = plt.figure(figsize=(14, 5))
gs_fig = gridspec.GridSpec(1, 3, figure=fig, wspace=0.35)

ax1 = fig.add_subplot(gs_fig[0, 0])
ax2 = fig.add_subplot(gs_fig[0, 1])
ax3 = fig.add_subplot(gs_fig[0, 2])

# 3.1 MARL learning curve
if len(marl_rewards) > 5:
    sm = smooth(marl_rewards)
    ax1.plot(marl_rewards, alpha=0.2, color=PALETTE["MARL"])
    ax1.plot(sm, color=PALETTE["MARL"], lw=2, label="MARL")
ax1.set_title("MARL Learning Curve", fontsize=11, fontweight="bold")
ax1.set_xlabel("Episode")
ax1.set_ylabel("Mean Episode Reward")
ax1.legend()

# 3.2 NARL-DGM meta / proxy reward dynamics
if len(narl_rewards) > 5:
    sm2 = smooth(narl_rewards)
    ax2.plot(narl_rewards, alpha=0.2, color=PALETTE["NARL-DGM"])
    ax2.plot(sm2, color=PALETTE["NARL-DGM"], lw=2, label="NARL proxy reward")

ax2.set_title("NARL-DGM Meta Dynamics", fontsize=11, fontweight="bold")
ax2.set_xlabel("Meta Episode")
ax2.legend()

# 3.3 Adaptive reward weights evolution (по логам META_LOG)
comp_labels = list(REW_IDX.keys())
if len(df_narl) > 1:
    # если логировали веса по компонентам отдельно
    weight_cols = [
        "w_growth", "w_stability", "w_inclusion",
        "w_resilience", "w_governance", "w_sustain", "w_explore",
    ]
    present_cols = [c for c in weight_cols if c in df_narl.columns]
    if present_cols:
        for c, lbl in zip(present_cols, comp_labels):
            ax3.plot(df_narl[c].values, lw=1.2, label=lbl, alpha=0.85)
        ax3.set_title("Adaptive Reward Weights Evolution", fontsize=11, fontweight="bold")
        ax3.set_xlabel("Meta Episode")
        ax3.set_ylabel("Weight")
        ax3.legend(fontsize=6, loc="upper right")
    else:
        ax3.text(
            0.5, 0.5,
            "Reward weights\nnot logged",
            ha="center", va="center",
            transform=ax3.transAxes, fontsize=10,
        )
        ax3.set_title("Adaptive Reward Weights", fontsize=11, fontweight="bold")
else:
    ax3.text(
        0.5, 0.5,
        "Weight evolution\n(insufficient data)",
        ha="center", va="center",
        transform=ax3.transAxes, fontsize=10,
    )
    ax3.set_title("Adaptive Reward Weights", fontsize=11, fontweight="bold")

plt.suptitle("Experiment Learning Dynamics", fontsize=13, fontweight="bold", y=1.02)
ARTS.save_fig(fig, "fig3_learning_curves")
plt.show()


# ── Figure 4: Scenario outcome comparison (Level 3) ───────────────────────

# Используем результаты трёхуровневого запуска, если они есть (через result)
try:
    scen_results = result["scenario_rollouts"]
except Exception:
    try:
        scen_results = result.scenario_rollouts
    except Exception:
        scen_results = {}

if scen_results:
    # В нашей новой 14.1 мы кладём df_narl_rollouts под ключ "raw"
    if isinstance(scen_results, dict) and "raw" in scen_results:
        df_sc = scen_results["raw"]
        # агрегируем по сценарию
        grp = df_sc.groupby("scenario")
        scen_names = list(grp.groups.keys())
        R_means = grp["mean_shaped_reward"].mean().values
        crisis_rates = grp["crisis_rate"].mean().values
    else:
        # fallback на старую структуру result.scenario_rollouts[s]["stats"]["by_scenario"]
        scen_names = list(scen_results.keys())
        R_means = []
        crisis_rates = []
        for s in scen_names:
            df_s = scen_results[s]["stats"]["by_scenario"]
            R_means.append(float(df_s["R_sum"].iloc[0]))
            crisis_rates.append(float(df_s["crisis_rate"].iloc[0]))
        R_means = np.asarray(R_means, dtype=float)
        crisis_rates = np.asarray(crisis_rates, dtype=float)

    fig, ax = plt.subplots(figsize=(10, 5))

    x = np.arange(len(scen_names))
    w = 0.35

    ax.bar(
        x - w / 2, R_means, width=w,
        color=PALETTE["NARL-DGM"], alpha=0.85,
        edgecolor="white", label="Mean shaped return",
    )
    ax2b = ax.twinx()
    ax2b.plot(
        x + w / 2, crisis_rates,
        color="#f18f01",
        marker="o", lw=1.5, label="Crisis rate",
    )

    ax.set_xticks(x)
    ax.set_xticklabels([n.replace(" ", "\n") for n in scen_names], fontsize=9)
    ax.set_ylabel("Mean shaped return", fontsize=11)
    ax2b.set_ylabel("Crisis rate", fontsize=11)
    ax.set_title("Scenario Outcomes under NARL-DGM", fontsize=12, fontweight="bold")

    # Легенды объединяем
    h1, l1 = ax.get_legend_handles_labels()
    h2, l2 = ax2b.get_legend_handles_labels()
    ax2b.legend(h1 + h2, l1 + l2, fontsize=9, loc="best")

    plt.tight_layout()
    ARTS.save_fig(fig, "fig4_scenario_outcomes")
    plt.show()


# ── Figure 5: Final adaptive reward weights ───────────────────────────────

final_w_arr = ADAPT_REWARD.weights.copy()
fig, ax = plt.subplots(figsize=(7, 5))
bars = ax.barh(
    comp_labels, final_w_arr,
    color=sns.color_palette("tab10", len(comp_labels)),
)
ax.axvline(
    1.0 / len(comp_labels),
    color="gray",
    ls="--", lw=1, label="Uniform prior",
)
ax.set_xlabel("Adaptive reward weight", fontsize=11)
ax.set_title(
    "NARL-DGM: Final Adaptive Reward Weights",
    fontsize=12, fontweight="bold",
)
ax.legend(fontsize=9)
plt.tight_layout()
ARTS.save_fig(fig, "fig5_adaptive_weights")
plt.show()


# ── Figure 6: Summary comparison bar chart ────────────────────────────────

fig, ax = plt.subplots(figsize=(8, 4))
models = list(df_cmp.index)
means  = df_cmp["Final 10% Avg"].values
stds   = df_cmp["Std Reward"].values
colors = [PALETTE.get(m.split()[0], "#888") for m in models]

bars = ax.bar(
    models, means, yerr=stds, capsize=6,
    color=colors, alpha=0.85, edgecolor="white",
    error_kw={"elinewidth": 1.5},
)
ax.set_ylabel("Final 10% Average Reward (top-k best)", fontsize=11)
ax.set_title(
    "Final Performance: Baseline vs MARL vs NARL-DGM",
    fontsize=12, fontweight="bold",
)

for bar, val in zip(bars, means):
    ax.text(
        bar.get_x() + bar.get_width() / 2.0,
        bar.get_height() + 0.005,
        f"{val:.3f}",
        ha="center", va="bottom",
        fontsize=9, fontweight="bold",
    )

plt.tight_layout()
ARTS.save_fig(fig, "fig6_final_comparison")
plt.show()

print("\n✅ All figures and comparison table saved")


═══ Experiment Comparison ══════════════════════════════
                Mean Reward  Std Reward  Max Reward  Final 10% Avg  Episodes
Model                                                                       
Baseline (VAR)      -5.3532      7.9641     -0.1931        -0.2313        20
MARL (MAPPO)        -1.4799      0.5479     -0.4677        -0.8595      1847
NARL-DGM            -0.1483      0.1577      0.0287         0.0244      1200

Wilcoxon test (MARL < NARL-DGM): stat=0.00, p=0.0000
  ✅ Significant (α=0.05)
   📄 Table saved  → table2_experiment_comparison.csv
   🖼  Figure saved → fig3_learning_curves.pdf
   🖼  Figure saved → fig4_scenario_outcomes.pdf
   🖼  Figure saved → fig5_adaptive_weights.pdf
   🖼  Figure saved → fig6_final_comparison.pdf

✅ All figures and comparison table saved


## 💾 Timing Summary & Artifact Download

In [23]:
# ─── CELL 16: FINAL ARCHIVE AND ARTIFACT SUMMARY ──────────────────────────

print("\n═══ Building experiment artifact archive ═════════════════════════════")

# 1. Никаких дополнительных реестров не трогаем — ARTS.save_fig / save_table
#    уже сохранили все файлы в CFG.OUTPUT_DIR. Здесь просто собираем архив.

archive_path = ARTS.build_archive()
print(f"\n📦 Archive built at: {archive_path}")

# 2. Печатаем краткий отчёт по файлам в OUTPUT_DIR
out_dir = Path(CFG.OUTPUT_DIR)
print("\n═══ Files in OUTPUT_DIR ══════════════════════════════════════════════")
if out_dir.exists():
    for p in sorted(out_dir.iterdir()):
        if p.is_file():
            print(f"  • {p.name}")
else:
    print("  (OUTPUT_DIR does not exist)")

print("\n✅ Experiment artifacts archived and ready for publication use.")


═══ Building experiment artifact archive ═════════════════════════════

📦 Archive ready → /content/foresight_results.zip

📦 Archive built at: /content/foresight_results.zip

═══ Files in OUTPUT_DIR ══════════════════════════════════════════════
  • fig1_eda_overview.pdf
  • fig1_eda_overview.png
  • fig2_baseline_scenarios_41d.pdf
  • fig2_baseline_scenarios_41d.png
  • fig3_learning_curves.pdf
  • fig3_learning_curves.png
  • fig4_scenario_outcomes.pdf
  • fig4_scenario_outcomes.png
  • fig5_adaptive_weights.pdf
  • fig5_adaptive_weights.png
  • fig6_final_comparison.pdf
  • fig6_final_comparison.png
  • log_baseline.csv
  • log_marl.csv
  • log_narl_dgm.csv
  • table2_experiment_comparison.csv
  • table2_experiment_comparison.tex
  • table_baseline_country_scenarios.csv
  • table_baseline_country_screen.csv
  • table_baseline_scenario_profiles.csv
  • table_crisis_prevalence.csv
  • table_narl_scenarios.csv
  • table_narl_scenarios.tex

✅ Experiment artifacts archived and ready for 

In [24]:
# ─── CELL 16.1: EXPORT TABLES TO EXCEL + ZIP FIGURES ─────────────────────

import os
import json
import zipfile
from pathlib import Path
from datetime import datetime

print("\n═══ Building consolidated Excel files and figures archive ═══════════")

# Базовая директория для сводных экспортов
base_dir = Path("artifacts")
base_dir.mkdir(parents=True, exist_ok=True)

# Директория, куда основной pipeline сохраняет артефакты
out_dir = Path(CFG.OUTPUT_DIR)
out_dir.mkdir(parents=True, exist_ok=True)

# Пытаемся подобрать Excel engine
excel_engine = None
for eng in ["openpyxl", "xlsxwriter"]:
    try:
        __import__(eng)
        excel_engine = eng
        break
    except Exception:
        pass

if excel_engine is None:
    print("⚠ Excel engine not available (neither openpyxl nor xlsxwriter).")
    print("  Fallback: Excel files will be skipped; CSV/ZIP/JSON will still be created.")
else:
    print(f"✅ Excel engine detected: {excel_engine}")

# 1. MASTER EXCEL: финальные таблицы для статьи ----------------------------

master_excel_path = base_dir / "foresight_results_master_tables.xlsx"

# Берём все CSV-таблицы из output/
table_files = sorted(out_dir.glob("table*.csv"))

if excel_engine is not None:
    with pd.ExcelWriter(master_excel_path, engine=excel_engine) as writer:
        if table_files:
            for fpath in table_files:
                try:
                    df = pd.read_csv(fpath)
                except Exception as e:
                    print(f"  ! Failed to read table {fpath.name}: {e}")
                    continue
                sheet_name = fpath.stem[:31]  # ограничение Excel
                df.to_excel(writer, sheet_name=sheet_name, index=False)
        else:
            pd.DataFrame({"msg": ["no table*.csv files found in output/"]}).to_excel(
                writer, sheet_name="info", index=False
            )
    print(f"\n📊 Master tables Excel saved to: {master_excel_path}")
else:
    print("\n📊 Master tables Excel skipped (no Excel engine available)")

# 2. LONG-FORM EXCEL: данные для построения графиков -----------------------

long_excel_path = base_dir / "foresight_results_plot_data.xlsx"

if excel_engine is not None:
    with pd.ExcelWriter(long_excel_path, engine=excel_engine) as writer:
        # MARL log
        try:
            df_marl = LOG_MARL.to_df()
            df_marl.to_excel(writer, sheet_name="marl_learning", index=False)
        except Exception as e:
            print("  ! MARL log export skipped:", str(e))

        # NARL meta log
        try:
            df_meta = META_LOG.to_df()
            df_meta.to_excel(writer, sheet_name="narl_meta", index=False)
        except Exception as e:
            print("  ! NARL meta log export skipped:", str(e))

        # Baseline log: сначала новый объект LOGBASE, потом старый LOG_BASEVAR
        try:
            df_base = LOGBASE.to_df()
            df_base.to_excel(writer, sheet_name="baseline_var", index=False)
        except Exception:
            try:
                df_base = LOG_BASEVAR.to_df()
                df_base.to_excel(writer, sheet_name="baseline_var", index=False)
            except Exception as e:
                print("  ! Baseline VAR log export skipped:", str(e))

    print(f"📈 Plot-data Excel saved to: {long_excel_path}")
else:
    print("📈 Plot-data Excel skipped (no Excel engine available)")

# 3. ZIP с графиками -------------------------------------------------------

fig_zip_path = base_dir / "foresight_figures.zip"
figure_files = sorted(out_dir.glob("fig*.png")) + sorted(out_dir.glob("fig*.pdf"))

with zipfile.ZipFile(fig_zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    if figure_files:
        for full_path in figure_files:
            if full_path.exists():
                zf.write(full_path, arcname=full_path.name)
    else:
        print("  (no fig*.png / fig*.pdf files found in output/, zip will be empty)")

print(f"🖼 Figures ZIP archive saved to: {fig_zip_path}")

# 4. JSON с конфигурацией запуска и seed'ами -------------------------------

run_meta = {}
run_meta["created_at_utc"] = datetime.utcnow().isoformat() + "Z"

try:
    cfg_dict = {}
    for k in dir(CFG):
        if k.startswith("_"):
            continue
        v = getattr(CFG, k)
        if isinstance(v, (int, float, bool, str)):
            cfg_dict[k] = v
        elif isinstance(v, (list, tuple)) and len(v) <= 32:
            cfg_dict[k] = list(v)
    run_meta["CFG"] = cfg_dict
except Exception as e:
    run_meta["CFG_error"] = str(e)

for name in ["SEED", "PY_SEED", "NP_SEED", "TORCH_SEED"]:
    if hasattr(CFG, name):
        run_meta[name] = getattr(CFG, name)

for attr in ["CODE_VERSION", "GIT_COMMIT", "EXPERIMENT_NAME"]:
    if hasattr(CFG, attr):
        run_meta[attr] = getattr(CFG, attr)

run_cfg_path = base_dir / "foresight_run_config.json"
with open(run_cfg_path, "w", encoding="utf-8") as f:
    json.dump(run_meta, f, ensure_ascii=False, indent=2)

print(f"\n🧾 Run configuration JSON saved to: {run_cfg_path}")

# 5. Итоговый сводный вывод ------------------------------------------------

print("\n═══ Consolidated files for interpretation ═══════════════════════════")
if excel_engine is not None:
    print(f"  • Master tables: {master_excel_path}")
    print(f"  • Plot data:     {long_excel_path}")
else:
    print("  • Master tables: skipped (no Excel engine)")
    print("  • Plot data:     skipped (no Excel engine)")
print(f"  • Figures ZIP:   {fig_zip_path}")
print(f"  • Run config:    {run_cfg_path}")

# Дополнительно покажем, что вообще есть в output/
print("\n═══ Files currently in output/ ══════════════════════════════════════")
for p in sorted(out_dir.iterdir()):
    if p.is_file():
        print(f"  • {p.name}")

print("\n✅ All consolidated exports and run config saved.")


═══ Building consolidated Excel files and figures archive ═══════════
✅ Excel engine detected: openpyxl

📊 Master tables Excel saved to: artifacts/foresight_results_master_tables.xlsx
📈 Plot-data Excel saved to: artifacts/foresight_results_plot_data.xlsx
🖼 Figures ZIP archive saved to: artifacts/foresight_figures.zip

🧾 Run configuration JSON saved to: artifacts/foresight_run_config.json

═══ Consolidated files for interpretation ═══════════════════════════
  • Master tables: artifacts/foresight_results_master_tables.xlsx
  • Plot data:     artifacts/foresight_results_plot_data.xlsx
  • Figures ZIP:   artifacts/foresight_figures.zip
  • Run config:    artifacts/foresight_run_config.json

═══ Files currently in output/ ══════════════════════════════════════
  • fig1_eda_overview.pdf
  • fig1_eda_overview.png
  • fig2_baseline_scenarios_41d.pdf
  • fig2_baseline_scenarios_41d.png
  • fig3_learning_curves.pdf
  • fig3_learning_curves.png
  • fig4_scenario_outcomes.pdf
  • fig4_scenario_o

In [25]:
# ─── CELL 17.0: AUTO-DOWNLOAD ALL ARTIFACTS (ZIP) ────────────────────────
# Собираем все ключевые артефакты в один ZIP и
# готовим автоскачивание (Colab) или ссылку (Jupyter).

import os
import zipfile
from pathlib import Path

# Базовая директория артефактов (из Cell 16.1)
base_dir = Path(ARTS.base_dir) if hasattr(ARTS, "base_dir") else Path("artifacts")
base_dir.mkdir(parents=True, exist_ok=True)

# Имя итогового ZIP с "всеми-всеми" результатами
final_zip_path = base_dir / "foresight_all_artifacts.zip"

# Список файлов, которые точно хотим внутри
files_to_pack = []

# 1) Архив из ARTS (Cell 16)
#    foresight_results.zip лежит в CFG.OUTPUT_DIR
arts_zip = Path(CFG.OUTPUT_DIR) / "foresight_results.zip"
if arts_zip.exists():
    files_to_pack.append(arts_zip)

# 2) Файлы из Cell 16.1
master_excel_path = base_dir / "foresight_results_master_tables.xlsx"
plot_excel_path   = base_dir / "foresight_results_plot_data.xlsx"
fig_zip_path      = base_dir / "foresight_figures.zip"
run_cfg_path      = base_dir / "foresight_run_config.json"

for p in [master_excel_path, plot_excel_path, fig_zip_path, run_cfg_path]:
    if p.exists():
        files_to_pack.append(p)

# 3) Страновые отчёты из Cells 17–18 (если уже были сгенерированы)
extra_files = [
    Path(CFG.OUTPUT_DIR) / "country_report_components.csv",
    Path(CFG.OUTPUT_DIR) / "country_report_summary.csv",
    Path(CFG.OUTPUT_DIR) / "country_report_components.xlsx",
]
for p in extra_files:
    if p.exists():
        files_to_pack.append(p)

# Собираем всё в один ZIP
with zipfile.ZipFile(final_zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for p in files_to_pack:
        p = Path(p)
        if not p.exists():
            continue
        # В архиве сохраняем только имя файла (без пути)
        zf.write(p, arcname=p.name)

print("\n═══ Unified artifact ZIP ═══════════════════════════════")
if files_to_pack:
    print("В ZIP включены файлы:")
    for p in files_to_pack:
        print(f"  • {Path(p).name}")
else:
    print("  (не найдено ни одного файла — ZIP будет пустым)")
print(f"\n📦 Unified ZIP created at: {final_zip_path}")

# Попытка автоскачивания в Google Colab
try:
    from google.colab import files as colab_files  # type: ignore
    print("\nОбнаружена среда Google Colab — инициируем скачивание ZIP…")
    colab_files.download(str(final_zip_path))
except Exception:
    # Если Colab недоступен, показываем HTML-ссылку (для обычного Jupyter)
    try:
        from IPython.display import FileLink, display
        print("\nНе Colab. Для скачивания нажмите на ссылку ниже:")
        display(FileLink(str(final_zip_path)))
    except Exception as e:
        print("\n⚠ Не удалось автоматически подготовить ссылку для скачивания:", e)

print("\n✅ Unified artifact ZIP ready for manual or auto download.")


═══ Unified artifact ZIP ═══════════════════════════════
В ZIP включены файлы:
  • foresight_results_master_tables.xlsx
  • foresight_results_plot_data.xlsx
  • foresight_figures.zip
  • foresight_run_config.json

📦 Unified ZIP created at: artifacts/foresight_all_artifacts.zip

Обнаружена среда Google Colab — инициируем скачивание ZIP…


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


✅ Unified artifact ZIP ready for manual or auto download.


In [26]:
# ─── CELL 17: INTERACTIVE COUNTRY REPORT (BASELINE · MARL · NARL-DGM) ────
# Универсальный отчёт по стране/странам:
# - меню выбора стран по названию,
# - таблицы и графики для Baseline VAR, MARL и NARL-DGM.

from typing import List, Dict
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

sns.set_style("whitegrid")


# 17.1 Вспомогательные маппинги страна <-> код --------------------------------
# Ожидаем, что в panel есть столбцы country_code и country_name (или name).
if "country_name" in panel.columns:
    COUNTRY_NAME_COL = "country_name"
elif "country" in panel.columns:
    COUNTRY_NAME_COL = "country"
else:
    COUNTRY_NAME_COL = "country_code"  # fallback

# список уникальных стран для меню
countries_df = (
    panel[[ "country_code", COUNTRY_NAME_COL ]]
    .drop_duplicates()
    .sort_values(COUNTRY_NAME_COL)
    .reset_index(drop=True)
)

code2name = dict(zip(countries_df["country_code"], countries_df[COUNTRY_NAME_COL]))
name2code = dict(zip(countries_df[COUNTRY_NAME_COL], countries_df["country_code"]))


print("Доступные страны (по названиям):")
for i, row in countries_df.iterrows():
    print(f"  {i:3d} | {row[COUNTRY_NAME_COL]} ({row['country_code']})")

print("\nВы можете ввести либо индексы через запятую (например: 0,5,10),")
print("либо сами названия стран через запятую (как в списке выше).")

user_input = input("\nВведите выбор стран: ").strip()

selected_codes: List[str] = []

if user_input:
    parts = [p.strip() for p in user_input.split(",") if p.strip()]
    for p in parts:
        # сначала пробуем интерпретировать как индекс
        if p.isdigit():
            idx = int(p)
            if 0 <= idx < len(countries_df):
                selected_codes.append(countries_df.loc[idx, "country_code"])
        else:
            # иначе как название страны
            if p in name2code:
                selected_codes.append(name2code[p])
            else:
                # пробуем case-insensitive
                matches = [n for n in name2code.keys() if n.lower() == p.lower()]
                if matches:
                    selected_codes.append(name2code[matches[0]])

selected_codes = sorted(set(selected_codes))

if not selected_codes:
    print("\n⚠ Страны не распознаны. Используем первую страну для примера.")
    selected_codes = [countries_df.loc[0, "country_code"]]

print("\nВыбраны страны:")
for cc in selected_codes:
    print(f"  {cc} — {code2name.get(cc, cc)}")


# 17.2 Baseline: извлечение прогнозов и ошибок по странам --------------------

baseline_country_to_stats: Dict[str, Dict] = {}

# baseline_preds, baseline_actuals, baseline_countries должны быть
# сформированы в Cell 8.
if (
    "baseline_preds" in globals()
    and "baseline_actuals" in globals()
    and "baseline_countries" in globals()
):
    for cc in selected_codes:
        idxs = [i for i, c in enumerate(baseline_countries) if c == cc]
        if not idxs:
            continue
        # Возьмём первый (или последний) набор прогнозов для этой страны
        i0 = idxs[-1]
        fc = baseline_preds[i0]    # (T,D)
        act = baseline_actuals[i0] # (T,D)
        mae = mean_absolute_error(act.ravel(), fc.ravel())
        rmse = np.sqrt(mean_squared_error(act.ravel(), fc.ravel()))
        baseline_country_to_stats[cc] = {
            "forecast": fc,
            "actual": act,
            "mae": mae,
            "rmse": rmse,
        }
else:
    print("\n⚠ baseline_* объекты не найдены — Baseline-часть отчёта будет пустой.")


# 17.3 MARL: эпизодические reward'ы (глобально, не по странам) --------------

df_marl = LOG_MARL.to_df() if len(LOG_MARL.history) else pd.DataFrame()
marl_rewards = df_marl["reward"].values.astype(float) if "reward" in df_marl.columns else np.array([])


# 17.4 NARL-DGM: meta / shaped rewards (глобально) --------------------------

df_narl = META_LOG.to_df() if META_LOG.history else pd.DataFrame()
if "shaped_reward" in df_narl.columns:
    narl_rewards = df_narl["shaped_reward"].values.astype(float)
elif "meta_loss" in df_narl.columns:
    narl_rewards = -df_narl["meta_loss"].values.astype(float)
else:
    narl_rewards = np.array([])


# 17.5 Country-level time series (исходные признаки) ------------------------

def get_country_timeseries(cc: str) -> pd.DataFrame:
    df_c = panel[panel["country_code"] == cc].copy()
    df_c = df_c.sort_values("year")
    return df_c


# 17.6 Визуализация по каждой выбранной стране -----------------------------

for cc in selected_codes:
    cname = code2name.get(cc, cc)
    print("\n" + "=" * 80)
    print(f"COUNTRY REPORT: {cname} ({cc})")
    print("=" * 80)

    df_c = get_country_timeseries(cc)
    years = df_c["year"].values

    # ---- Табличка краткой baseline-статистики -------------------------
    if cc in baseline_country_to_stats:
        st = baseline_country_to_stats[cc]
        print("\nBaseline VAR metrics (unified 41D):")
        print(f"  MAE  : {st['mae']:.4f}")
        print(f"  RMSE : {st['rmse']:.4f}")
    else:
        print("\nBaseline VAR metrics: нет данных для этой страны (либо не прошла safety screen).")

    # ---- Фигуры: временные ряды + baseline прогноз -------------------
    # Выберем пару ключевых индикаторов, если есть
    key_feats = [
        "gdp_growth", "gdp_pc_growth", "inflation",
        "unemployment", "rd_gdp", "internet_users",
    ]
    key_feats = [f for f in key_feats if f in df_c.columns]

    n_rows = 2
    n_cols = 2
    fig_ts, axes_ts = plt.subplots(
        n_rows, n_cols, figsize=(10, 6), sharex=True
    )
    axes_ts = axes_ts.ravel()

    for i, feat in enumerate(key_feats[: n_rows * n_cols]):
        ax = axes_ts[i]
        ax.plot(years, df_c[feat].values, label=feat, color="#2e86ab")
        ax.set_title(feat, fontsize=9)
        ax.grid(True, alpha=0.3)
        ax.legend(fontsize=8)

    for j in range(len(key_feats), n_rows * n_cols):
        axes_ts[j].axis("off")

    fig_ts.suptitle(f"{cname}: key indicators over time", fontsize=12, fontweight="bold")
    fig_ts.tight_layout()
    plt.show()

    # ---- Фигура: baseline forecast vs actual (по г/в среднему) --------
    if cc in baseline_country_to_stats:
        st = baseline_country_to_stats[cc]
        fc = st["forecast"]  # (T,D)
        act = st["actual"]   # (T,D)
        horizon = fc.shape[0]
        t_idx = np.arange(horizon)

        fig_b, ax_b = plt.subplots(figsize=(8, 4))
        # усреднение по фичам, чтобы получить общую траекторию
        ax_b.plot(t_idx, act.mean(axis=1), "-o", label="Actual (mean over features)", color="#7fb3d3")
        ax_b.plot(t_idx, fc.mean(axis=1), "-s", label="Forecast (mean over features)", color="#e63946")
        ax_b.set_xlabel("Forecast step")
        ax_b.set_ylabel("Scaled value (mean over 41D)")
        ax_b.set_title(f"{cname}: Baseline VAR forecast vs actual (mean over 41D)")
        ax_b.legend(fontsize=9)
        ax_b.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()

    # ---- Фигуры: MARL и NARL глобальные кривые (для контекста) --------
    fig_m, axes_m = plt.subplots(1, 2, figsize=(12, 4))

    # MARL reward curve
    ax_m1 = axes_m[0]
    if marl_rewards.size > 5:
        sm = smooth(marl_rewards)
        ax_m1.plot(marl_rewards, alpha=0.2, color="#2e86ab", label="MARL reward")
        ax_m1.plot(sm, color="#2e86ab", lw=2, label="MARL smoothed")
    ax_m1.set_title("MARL learning (global)", fontsize=11, fontweight="bold")
    ax_m1.set_xlabel("Episode")
    ax_m1.set_ylabel("Reward")
    ax_m1.legend(fontsize=8)
    ax_m1.grid(True, alpha=0.3)

    # NARL shaped reward curve
    ax_m2 = axes_m[1]
    if narl_rewards.size > 5:
        sm2 = smooth(narl_rewards)
        ax_m2.plot(narl_rewards, alpha=0.2, color="#e63946", label="NARL shaped")
        ax_m2.plot(sm2, color="#e63946", lw=2, label="NARL smoothed")
    ax_m2.set_title("NARL-DGM meta (global)", fontsize=11, fontweight="bold")
    ax_m2.set_xlabel("Meta episode")
    ax_m2.set_ylabel("Shaped reward / proxy")
    ax_m2.legend(fontsize=8)
    ax_m2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

print("\n✅ Country report(s) generated.")

Доступные страны (по названиям):
    0 | Argentina (ARG)
    1 | Austria (AUT)
    2 | Belarus (BLR)
    3 | Belgium (BEL)
    4 | Bulgaria (BGR)
    5 | Canada (CAN)
    6 | Colombia (COL)
    7 | Costa Rica (CRI)
    8 | Croatia (HRV)
    9 | Cyprus (CYP)
   10 | Czechia (CZE)
   11 | Denmark (DNK)
   12 | Estonia (EST)
   13 | Finland (FIN)
   14 | France (FRA)
   15 | Georgia (GEO)
   16 | Germany (DEU)
   17 | Greece (GRC)
   18 | Hungary (HUN)
   19 | Iceland (ISL)
   20 | Indonesia (IDN)
   21 | Ireland (IRL)
   22 | Israel (ISR)
   23 | Italy (ITA)
   24 | Kazakhstan (KAZ)
   25 | Korea, Rep. (KOR)
   26 | Kyrgyz Republic (KGZ)
   27 | Latvia (LVA)
   28 | Lithuania (LTU)
   29 | Malaysia (MYS)
   30 | Malta (MLT)
   31 | Mexico (MEX)
   32 | Moldova (MDA)
   33 | Netherlands (NLD)
   34 | Norway (NOR)
   35 | Panama (PAN)
   36 | Poland (POL)
   37 | Portugal (PRT)
   38 | Romania (ROU)
   39 | Russian Federation (RUS)
   40 | Slovak Republic (SVK)
   41 | Slovenia (SVN)
   42

In [27]:
# ─── CELL 18: DEEP COUNTRY DECOMPOSITION & EXPORT ─────────────────────────
# Углублённый отчёт по странам:
# - декомпозиция NARL-компонент по странам;
# - сравнение Baseline · MARL · NARL агрегированных метрик;
# - автоматическое сохранение данных по странам (CSV + Excel при наличии движка).

from typing import Dict, List
import seaborn as sns
import matplotlib.pyplot as plt
from pathlib import Path

sns.set_style("whitegrid")

OUTPUT_DIR = Path(CFG.OUTPUT_DIR)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# 18.1 Проверяем, что есть selected_codes из Cell 17 ------------------------

if "selected_codes" not in globals() or not selected_codes:
    print("⚠ selected_codes не найден или пуст. Сначала выполните Cell 17 (Country report).")
    raise SystemExit

# 18.2 Функции для страновой компоненты NARL и агрегатов -------------------

def country_component_snapshot(cc: str) -> Dict[str, float]:
    """
    Делает моментальный snapshot компонентной NARL-награды для страны cc
    на основе последней доступной строки panel.
    """
    df_c = panel[panel["country_code"] == cc].copy()
    if df_c.empty:
        return {name: 0.0 for name in REW_IDX.keys()}

    df_c = df_c.sort_values("year")
    row = df_c.iloc[-1]

    summary = {
        "gdp_growth": float(row.get("gdp_growth", 0.0)),
        "gdp_pc_usd": float(row.get("gdp_pc_usd", 0.0)),
        "rd_gdp": float(row.get("rd_gdp", 0.0)),
        "inflation": float(row.get("inflation", 0.0)),
        "unemployment": float(row.get("unemployment", 0.0)),
        "gdp_growth_vol3": 0.0,
        "gini_index": float(row.get("gini_index", 0.0)),
        "school_tertiary": float(row.get("school_tertiary", 0.0)),
        "crisis_score": 0.0,
        "reserves": 0.0,
        "wgi_govt_eff": float(row.get("wgi_govt_eff", 0.0)),
        "wgi_rule_law": float(row.get("wgi_rule_law", 0.0)),
        "wgi_corruption": float(row.get("wgi_corruption", 0.0)),
        "co2_pc": float(row.get("co2_pc", 0.0)),
        "renewable_energy": float(row.get("renewable_energy", 0.0)),
        "forest_area_pct": float(row.get("forest_area_pct", 0.0)),
        "energy_use_pc": float(row.get("energy_use_pc", 0.0)),
        "action_entropy": 0.0,
        "ai_switches": 0.0,
    }

    r_vec = decompose_reward({"state_summary": summary})
    return {name: float(r_vec[idx]) for name, idx in REW_IDX.items()}


def country_baseline_aggregate(cc: str) -> Dict[str, float]:
    """
    Возвращает агрегированные baseline-метрики по стране cc
    (MAE, RMSE, pseudo_reward).
    """
    if "baseline_country_to_stats" not in globals():
        return {
            "baseline_mae": float("nan"),
            "baseline_rmse": float("nan"),
            "baseline_reward": float("nan"),
        }
    if cc not in baseline_country_to_stats:
        return {
            "baseline_mae": float("nan"),
            "baseline_rmse": float("nan"),
            "baseline_reward": float("nan"),
        }

    st = baseline_country_to_stats[cc]
    mae = float(st["mae"])
    rmse = float(st["rmse"])
    return {
        "baseline_mae": mae,
        "baseline_rmse": rmse,
        "baseline_reward": -mae,
    }


def global_marl_narl_summary() -> Dict[str, float]:
    """
    Общие агрегаты по MARL и NARL (глобально).
    """
    df_marl = LOG_MARL.to_df() if len(LOG_MARL.history) else pd.DataFrame()
    df_narl = META_LOG.to_df() if META_LOG.history else pd.DataFrame()

    out = {}

    if "reward" in df_marl.columns and len(df_marl) > 0:
        r = df_marl["reward"].values.astype(float)
        out["marl_mean_reward"] = float(r.mean())
        out["marl_max_reward"] = float(r.max())
        out["marl_episodes"] = int(len(r))
    else:
        out["marl_mean_reward"] = float("nan")
        out["marl_max_reward"] = float("nan")
        out["marl_episodes"] = 0

    if "shaped_reward" in df_narl.columns and len(df_narl) > 0:
        r2 = df_narl["shaped_reward"].values.astype(float)
        out["narl_mean_reward"] = float(r2.mean())
        out["narl_max_reward"] = float(r2.max())
        out["narl_episodes"] = int(len(r2))
    elif "meta_loss" in df_narl.columns and len(df_narl) > 0:
        r2 = -df_narl["meta_loss"].values.astype(float)
        out["narl_mean_reward"] = float(r2.mean())
        out["narl_max_reward"] = float(r2.max())
        out["narl_episodes"] = int(len(r2))
    else:
        out["narl_mean_reward"] = float("nan")
        out["narl_max_reward"] = float("nan")
        out["narl_episodes"] = 0

    return out

# 18.3 Формируем таблицу компонент по странам --------------------------------

rows_comp: List[Dict] = []
rows_summary: List[Dict] = []

global_summary = global_marl_narl_summary()

for cc in selected_codes:
    cname = code2name.get(cc, cc)

    comp = country_component_snapshot(cc)
    base = country_baseline_aggregate(cc)

    row_c = {"country_code": cc, "country_name": cname}
    row_c.update(comp)
    rows_comp.append(row_c)

    row_s = {"country_code": cc, "country_name": cname}
    row_s.update(base)
    row_s.update(global_summary)
    rows_summary.append(row_s)

df_comp = pd.DataFrame(rows_comp)
df_summary = pd.DataFrame(rows_summary)

print("\n=== Country-level NARL component snapshot ===")
display(df_comp)

print("\n=== Country-level Baseline/MARL/NARL summary (aggregated) ===")
display(df_summary)

# 18.4 Сохранение данных по странам (CSV + Excel при наличии движка) -------

comp_csv_path = OUTPUT_DIR / "country_report_components.csv"
summary_csv_path = OUTPUT_DIR / "country_report_summary.csv"
comp_xlsx_path = OUTPUT_DIR / "country_report_components.xlsx"

df_comp.to_csv(comp_csv_path, index=False)
df_summary.to_csv(summary_csv_path, index=False)

excel_engine = None
for eng in ["openpyxl", "xlsxwriter"]:
    try:
        __import__(eng)
        excel_engine = eng
        break
    except Exception:
        pass

if excel_engine is not None:
    with pd.ExcelWriter(comp_xlsx_path, engine=excel_engine) as writer:
        df_comp.to_excel(writer, sheet_name="components", index=False)
        df_summary.to_excel(writer, sheet_name="summary", index=False)
    print("\n📁 Saved country-level data:")
    print(f"  • Components CSV : {comp_csv_path}")
    print(f"  • Summary CSV    : {summary_csv_path}")
    print(f"  • Excel (both)   : {comp_xlsx_path}")
else:
    print("\n⚠ Excel engine not available (neither openpyxl nor xlsxwriter).")
    print("  Excel export skipped; CSV files were still saved.")
    print("\n📁 Saved country-level data:")
    print(f"  • Components CSV : {comp_csv_path}")
    print(f"  • Summary CSV    : {summary_csv_path}")

# 18.5 Визуализации: компоненты NARL по странам -----------------------------

if not df_comp.empty:
    comp_cols = list(REW_IDX.keys())
    plot_df = df_comp.set_index("country_name")[comp_cols]

    plt.figure(figsize=(max(6, len(plot_df) * 0.5), 4))
    sns.heatmap(
        plot_df,
        annot=True, fmt=".2f",
        cmap="RdYlGn",
        center=0.0,
        cbar_kws={"label": "component score"},
    )
    plt.title("NARL-DGM components snapshot by country", fontsize=12, fontweight="bold")
    plt.xlabel("Component")
    plt.ylabel("Country")
    plt.tight_layout()
    plt.show()

    for _, row in df_comp.iterrows():
        cname = row["country_name"]
        vals = [row[c] for c in comp_cols]
        fig, ax = plt.subplots(figsize=(7, 4))
        ax.bar(comp_cols, vals, color=sns.color_palette("tab10", len(comp_cols)))
        ax.axhline(0.0, color="gray", lw=1)
        ax.set_ylabel("Component score", fontsize=11)
        ax.set_title(f"{cname}: NARL-DGM component profile", fontsize=12, fontweight="bold")
        plt.xticks(rotation=45, ha="right")
        plt.tight_layout()
        plt.show()

# 18.6 Визуализация: Baseline vs global MARL/NARL для стран ---------------

if not df_summary.empty:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    ax1, ax2 = axes

    ax1.scatter(
        df_summary["baseline_reward"],
        df_summary["marl_mean_reward"],
        color="#2e86ab", label="MARL",
    )
    ax1.scatter(
        df_summary["baseline_reward"],
        df_summary["narl_mean_reward"],
        color="#e63946", label="NARL-DGM",
        marker="s",
    )
    ax1.axhline(0.0, color="gray", lw=0.8)
    ax1.axvline(0.0, color="gray", lw=0.8)
    ax1.set_xlabel("Baseline pseudo-reward (-MAE)", fontsize=10)
    ax1.set_ylabel("Mean reward (global)", fontsize=10)
    ax1.set_title("Baseline vs MARL/NARL (mean)", fontsize=11, fontweight="bold")
    ax1.legend(fontsize=9)
    ax1.grid(True, alpha=0.3)

    ax2.scatter(
        df_summary["baseline_rmse"],
        df_summary["marl_max_reward"],
        color="#2e86ab", label="MARL",
    )
    ax2.scatter(
        df_summary["baseline_rmse"],
        df_summary["narl_max_reward"],
        color="#e63946", label="NARL-DGM",
        marker="s",
    )
    ax2.set_xlabel("Baseline RMSE", fontsize=10)
    ax2.set_ylabel("Max reward (global)", fontsize=10)
    ax2.set_title("Baseline RMSE vs MARL/NARL max reward", fontsize=11, fontweight="bold")
    ax2.legend(fontsize=9)
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

print("\n✅ Deep country decomposition report generated and saved.")


=== Country-level NARL component snapshot ===


,country_code,country_name,growth,stability,inclusion,resilience,governance,sustainability,exploration
0,ARG,Argentina,0.0,-71.862999,0.0,0.0,0.0,0.0,0.0
1,AUT,Austria,0.0,-0.320445,0.0,0.0,0.0,0.0,0.0
2,BLR,Belarus,0.0,0.136901,0.0,0.0,0.0,0.0,0.0
3,FIN,Finland,0.0,-0.163376,0.0,0.0,0.0,0.0,0.0
4,KAZ,Kazakhstan,0.0,-2.206466,0.0,0.0,0.0,0.0,0.0
5,POL,Poland,0.0,-0.965268,0.0,0.0,0.0,0.0,0.0
6,PRT,Portugal,0.0,-0.010935,0.0,0.0,0.0,0.0,0.0
7,RUS,Russian Federation,0.0,0.107992,0.0,0.0,0.0,0.0,0.0
8,THA,Thailand,0.0,0.025139,0.0,0.0,0.0,0.0,0.0



=== Country-level Baseline/MARL/NARL summary (aggregated) ===


,country_code,country_name,baseline_mae,baseline_rmse,baseline_reward,marl_mean_reward,marl_max_reward,marl_episodes,narl_mean_reward,narl_max_reward,narl_episodes
0,ARG,Argentina,NaN,NaN,NaN,-1.479913,-0.467704,1847,-0.148265,0.028695,1200
1,AUT,Austria,NaN,NaN,NaN,-1.479913,-0.467704,1847,-0.148265,0.028695,1200
2,BLR,Belarus,NaN,NaN,NaN,-1.479913,-0.467704,1847,-0.148265,0.028695,1200
3,FIN,Finland,NaN,NaN,NaN,-1.479913,-0.467704,1847,-0.148265,0.028695,1200
4,KAZ,Kazakhstan,0.521891,0.984994,-0.521891,-1.479913,-0.467704,1847,-0.148265,0.028695,1200
5,POL,Poland,15.756525,103.167124,-15.756525,-1.479913,-0.467704,1847,-0.148265,0.028695,1200
6,PRT,Portugal,NaN,NaN,NaN,-1.479913,-0.467704,1847,-0.148265,0.028695,1200
7,RUS,Russian Federation,1.278633,2.962109,-1.278633,-1.479913,-0.467704,1847,-0.148265,0.028695,1200
8,THA,Thailand,0.434433,1.042319,-0.434433,-1.479913,-0.467704,1847,-0.148265,0.028695,1200



📁 Saved country-level data:
  • Components CSV : output/country_report_components.csv
  • Summary CSV    : output/country_report_summary.csv
  • Excel (both)   : output/country_report_components.xlsx

✅ Deep country decomposition report generated and saved.


In [28]:
# ─── CELL 19: PACKAGE AND DOWNLOAD ALL RESULTS ────────────────────────────

import zipfile
from pathlib import Path

print("\n═══ Packaging all experiment outputs for download ═══════════════════")

out_dir = Path(CFG.OUTPUT_DIR)
art_dir = Path("artifacts")
out_dir.mkdir(parents=True, exist_ok=True)
art_dir.mkdir(parents=True, exist_ok=True)

final_zip_path = art_dir / "foresight_all_artifacts.zip"

files_to_pack = []

# 1. Всё из output/
for p in sorted(out_dir.iterdir()):
    if p.is_file():
        files_to_pack.append(p)

# 2. Ключевые файлы из artifacts/
for p in sorted(art_dir.iterdir()):
    if p.is_file() and p.name != final_zip_path.name:
        files_to_pack.append(p)

# 3. Собираем единый ZIP
with zipfile.ZipFile(final_zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for p in files_to_pack:
        zf.write(p, arcname=p.name)

print(f"\n📦 Unified ZIP created: {final_zip_path}")
print("\nСодержимое ZIP:")
for p in files_to_pack:
    print(f"  • {p.name}")

# 4. Автоскачивание в Colab или ссылка в Jupyter
download_done = False

try:
    from google.colab import files as colab_files  # type: ignore
    print("\n✅ Google Colab detected — starting browser download...")
    colab_files.download(str(final_zip_path))
    download_done = True
except Exception:
    pass

if not download_done:
    try:
        from IPython.display import FileLink, display
        print("\nℹ Auto-download недоступен. Нажми на ссылку ниже для скачивания ZIP:")
        display(FileLink(str(final_zip_path)))
    except Exception as e:
        print("\n⚠ Не удалось создать ссылку на скачивание:", e)
        print(f"Скачай файл вручную через файловую панель: {final_zip_path}")

print("\n✅ Results are packaged and ready to download.")


═══ Packaging all experiment outputs for download ═══════════════════

📦 Unified ZIP created: artifacts/foresight_all_artifacts.zip

Содержимое ZIP:
  • country_report_components.csv
  • country_report_components.xlsx
  • country_report_summary.csv
  • fig1_eda_overview.pdf
  • fig1_eda_overview.png
  • fig2_baseline_scenarios_41d.pdf
  • fig2_baseline_scenarios_41d.png
  • fig3_learning_curves.pdf
  • fig3_learning_curves.png
  • fig4_scenario_outcomes.pdf
  • fig4_scenario_outcomes.png
  • fig5_adaptive_weights.pdf
  • fig5_adaptive_weights.png
  • fig6_final_comparison.pdf
  • fig6_final_comparison.png
  • log_baseline.csv
  • log_marl.csv
  • log_narl_dgm.csv
  • table2_experiment_comparison.csv
  • table2_experiment_comparison.tex
  • table_baseline_country_scenarios.csv
  • table_baseline_country_screen.csv
  • table_baseline_scenario_profiles.csv
  • table_crisis_prevalence.csv
  • table_narl_scenarios.csv
  • table_narl_scenarios.tex
  • foresight_figures.zip
  • foresight_res

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


✅ Results are packaged and ready to download.
